In [ ]:
import re
import os
import sqlite3
import time
import json
import ast
import pandas as pd
import lmstudio as lms
from IPython.display import display, Markdown, clear_output
import textwrap
import io
import matplotlib.pyplot as plt
from PIL import Image
from asma.core.parser import parse_bioc_to_llm_markdown


JSON_PATH = "d:/Codes/GitHub/asma-workspace/litsift/server/cache/pmc_fetches/401d6d3929e951aaf358dc49e42a73c957ac495eeea976cc71647b19543f3b38.json"
MODEL_NAME = "google/gemma-4-e2b-qat"
BASE_URL = "localhost:1234"
GT_PATH = "d:/Codes/GitHub/asma-workspace/AM-AS/extracted_results_multipass.csv"
OUTPUT_CSV_PATH = "d:/Codes/GitHub/asma-workspace/AM-AS/extracted_results_multipass.csv"

In [2]:
def get_paper_content_from_json(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        bioc_data = json.load(f)
    # The bioc_data is a list of documents. Parse it to markdown.
    return parse_bioc_to_llm_markdown(bioc_data)

paper_text = get_paper_content_from_json(JSON_PATH)
print(f"Loaded and parsed paper text: {len(paper_text)} characters.")


Loaded and parsed paper text: 61549 characters.


In [3]:
client = lms.Client(BASE_URL)
model = client.llm.model(MODEL_NAME)
chat = lms.Chat()

document_context = (
    "You are a document reading assistant. "
    "Below is the research article. "
    "Confirm you have loaded the document by responding with exactly the JSON array: ['Ready']:\n\n"
    f"<document>\n{paper_text}\n</document>"
)

# chat.add_system_prompt(document_context)
chat._messages[:] = chat._messages[:0]
chat.add_user_message(document_context)
pre_warm_reply = model.respond(chat)
chat._messages.append({"role": "assistant", "content": "['Ready']"})

Markdown(pre_warm_reply.content)

Thinking Process:

1.  **Analyze the Request:** The user wants me to confirm I have loaded the provided research article by responding with exactly the JSON array: `['Ready']`.

2.  **Analyze the Input:** The input is a research article (text) about lytic bacteriophages Sfin-2 and Sfin-6 infecting MDR *Shigella* spp. and their application on raw chicken.

3.  **Determine the Goal:** The task is simple confirmation of document loading. Since the document is present and I have processed it (as the assistant), the correct output is the specified JSON string.

4.  **Format the Output:** Ensure the output matches the strict requirement: `['Ready']`.

5.  **Final Verification:**
    *   Did I load the document? Yes.
    *   Am I responding with *exactly* the required JSON array? Yes.__LM_STUDIO_INTERNAL_LSEP_SYNTHETIC_REASONING_END_f4e9a8d2c6b14d0c9e5f3a7b8c1d2e6a__['Ready']

In [5]:
tbl_gt = pd.read_csv(GT_PATH)
tbl_extract = tbl_gt  #tbl_gt.iloc[[0]].copy()
blank_row = pd.DataFrame([[""] * len(tbl_extract.columns)], columns=tbl_extract.columns)
# tbl_extract = pd.concat([tbl_extract, blank_row], ignore_index=True)
tbl_extract

,Phage Name,Phage Genome Accession/Bioproject,Phage Genome size (bp),Phage GC content (%),Phage TEM shows structural similarity with,Phage TEM dimensions/Capsid morphology,Phage Taxonomy,Phage type: Lytic/ Lysogenic/ Engineered,Place of Sample collection,Phage isolation Sample,Studied Host Strain/ Propagating Bacterial Strain,Primary targeted bacteria species (not Strain),Phage's Plaque characteristics/Shape,Optimal MOI,Latent period (min),Burst size (phage/infected bacterium),Optimal Temperature (°C),Optimal pH
0,P2,OP797796,"42,660",58.8,podovirus,icosahedral head of diameter 54.4 – 1.5 nm (me...,"Caudoviricetes, Autographiviridae, Ahphunavirus",Lytic,"Fortis hospital, Mumbai, India",Sewage,A. hydrophila CECT 839T,Aeromonas hydrophila,clear circular plaques surrounded by a turbid ...,0.01,~90,34 ± 7,"4°C, 30°C, and 40°C",6 ‒ 10
1,Sfin-2,MK972831,"50,390 bp",44.9,NOT AVAILABLE,Isometric head (64.90 ± 2.04 nm) and non-contr...,"Siphoviridae, Caudovirales",Lytic,"Ganga river, near Barrackpore, North 24 Pargan...","Ganga river, near Barrackpore, North 24 Pargan...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
def find_first_empty_cell(df):
    for idx, row in df.iterrows():
        for col in df.columns:
            val = row[col]
            if pd.isna(val) or val == "" or val is None:
                return idx, col
    return None, None

def find_example_value(df, target_column):
    filled_values = df[
        df[target_column].notna() &
        (df[target_column] != "") &
        (df[target_column] != "Not specified")
    ][target_column]
    if not filled_values.empty:
        return filled_values.iloc[0]
    return None

def build_conditions_and_example(df):
    row_idx, target_column = find_first_empty_cell(df)
    if row_idx is None or target_column is None:
        return None, None, None, None
    row = df.loc[row_idx]
    filled_conditions = []
    for col in df.columns:
        if col != target_column:
            val = row[col]
            if not (pd.isna(val) or val == "" or val is None):
                filled_conditions.append(f"the '{col}' is '{val}'")
    example_val = find_example_value(df, target_column)
    return row_idx, target_column, filled_conditions, example_val

def format_extraction_prompt(target_column, conditions, example_val, template_str):
    conditions_str = ""
    if conditions:
        conditions_str = "When " + "\nand \n".join(conditions)
    example_str = ""
    if example_val:
        example_str = example_val
    return template_str.format(
        target_column=target_column,
        conditions=conditions_str,
        example_format=example_str
    ).strip()

def split_llm_response(text):
    end_pattern = r"(?:<channel\|>)|(?:<\/channel>)|(?:channel\|>)"
    matches = list(re.finditer(end_pattern, text, re.IGNORECASE))
    if matches:
        last_match = matches[-1]
        start, end = last_match.span()
        thought = text[:start]
        content = text[end:]
        start_pattern = r"(?:<\|)?channel>thought"
        thought = re.sub(start_pattern, "", thought, flags=re.IGNORECASE)
        return thought.strip(), content.strip()
    return "", text.strip()

def extract_values_from_decision(text):
    lines = text.split('\n')
    values_start = False
    values_list = []
    for line in lines:
        cleaned = line.strip()
        if not cleaned:
            continue
        if "values:" in cleaned.lower():
            values_start = True
            continue
        if values_start:
            val = cleaned
            val = re.sub(r'^V\d+:\s*', '', val, flags=re.IGNORECASE)
            if val.startswith(("-", "*", "•", "1.", "2.", "3.", "4.")):
                val = val.split(None, 1)[-1]
            values_list.append(val.strip().strip('"').strip("'"))
    if not values_list:
        for line in lines:
            cleaned = line.strip()
            if cleaned and not cleaned.lower().startswith("analysis:"):
                val = cleaned
                val = re.sub(r'^V\d+:\s*', '', val, flags=re.IGNORECASE)
                if val.startswith(("-", "*", "•")):
                    val = val.split(None, 1)[-1]
                values_list.append(val.strip().strip('"').strip("'"))
    return "\n".join(values_list)

def extract_json_array(text):
    text = text.strip()
    if text.startswith("```"):
        first_newline = text.find("\n")
        if first_newline != -1:
            text = text[first_newline:]
        if text.endswith("```"):
            text = text[:-3]
        text = text.strip()
        
    start_idx = text.find("[")
    end_idx = text.rfind("]")
    if start_idx == -1 or end_idx == -1 or end_idx < start_idx:
        raise ValueError("No valid JSON array boundaries found.")
        
    raw_array = text[start_idx:end_idx + 1]
    try:
        parsed = json.loads(raw_array)
        if isinstance(parsed, list):
            return [str(item) for item in parsed]
    except Exception:
        pass
    try:
        parsed = ast.literal_eval(raw_array)
        if isinstance(parsed, list):
            return [str(item) for item in parsed]
    except Exception:
        pass
    raise ValueError("Failed to parse string segment as a JSON list.")

def insert_and_split(df, row_idx, col_name, extracted_values):
    if not extracted_values:
        df.at[row_idx, col_name] = "Not specified"
        return df
    original_row = df.iloc[row_idx].copy()
    new_rows = []
    for val in extracted_values:
        row_copy = original_row.copy()
        row_copy[col_name] = val
        new_rows.append(row_copy)
    new_rows_df = pd.DataFrame(new_rows)
    df_before = df.iloc[:row_idx]
    df_after = df.iloc[row_idx + 1:]
    return pd.concat([df_before, new_rows_df, df_after], ignore_index=True)

def df_to_image(df):
    temp_df = df.fillna('-').replace('', '-')
    max_lens = []
    for col in temp_df.columns:
        max_len = max(temp_df[col].astype(str).map(len).max(), len(str(col)))
        max_lens.append(max_len)
    total_len = sum(max_lens)
    col_widths = [max(0.03, w / total_len) for w in max_lens]
    total_normalized = sum(col_widths)
    col_widths = [w / total_normalized for w in col_widths]
    
    wrapped_headers = []
    for col, width_ratio in zip(temp_df.columns, col_widths):
        wrap_width = max(12, int(width_ratio * 150))
        wrapped_headers.append(textwrap.fill(str(col), width=wrap_width))
        
    wrapped_data = []
    for row in temp_df.values:
        wrapped_row = []
        for val, width_ratio in zip(row, col_widths):
            wrap_width = max(12, int(width_ratio * 150))
            wrapped_row.append(textwrap.fill(str(val), width=wrap_width))
        wrapped_data.append(wrapped_row)
        
    row_height = 1.8
    fig_width = 30
    fig_height = len(df) * row_height + 4.0
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis('off')
    tbl = ax.table(
        cellText=wrapped_data,
        colLabels=wrapped_headers,
        colWidths=col_widths,
        loc='center',
        cellLoc='left'
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8.0)
    tbl.scale(1.0, 1.8)
    for (row_idx, col_idx), cell in tbl.get_celld().items():
        cell.set_edgecolor('#dcdcdc')
        if row_idx == 0:
            cell.set_text_props(weight='bold', color='white')
            cell.set_facecolor('#2b3e50')
        else:
            if row_idx % 2 == 0:
                cell.set_facecolor('#f8f9fa')
            else:
                cell.set_facecolor('#ffffff')
    buf = io.BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight', dpi=140)
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf)

In [7]:
system_prompt = (
    "**SYSTEM INSTRUCTIONS**\n\n"
    "You are an expert data extraction, database schema and normalization assistant specialized in systematic reviews.\n"
    "Your core task is to extract highly accurate, to-the-point variable values from the provided research papers.\n\n"
    "RULES:\n"
    "* Always execute internal reasoning step-by-step before arriving at a conclusion.\n"
    "* Rely strictly on the facts explicitly reported in the document. Do not guess, or hallucinate.\n\n"
    f"<document>\n{paper_text}\n</document>"
)

chat._messages[:] = chat._messages[:0]
chat.add_system_prompt(system_prompt)

print(chat)
Markdown(system_prompt)

Chat.from_history({
  "messages": [
    {
      "content": [
        {
          "text": "**SYSTEM INSTRUCTIONS**\n\nYou are an expert data extraction, database schema and normalization assistant specialized in systematic reviews.\nYour core task is to extract highly accurate, to-the-point variable values from the provided research papers.\n\nRULES:\n* Always execute internal reasoning step-by-step before arriving at a conclusion.\n* Rely strictly on the facts explicitly reported in the document. Do not guess, or hallucinate.\n\n<document>\n# Characterizations of novel broad-spectrum lytic bacteriophages Sfin-2 and Sfin-6 infecting MDR Shigella spp. with their application on raw chicken to reduce the Shigella load\n\nDOI: https://doi.org/10.3389/fmicb.2023.1240570\nPMC: 10716491\nPMID: 38094623\nPublished: Volume 14, (2023)\nAuthors: S. K. Tousif Ahamed, Srijana Rai, Chiranjib Guin, Rameez Moidu Jameela, Somasri Dam, Dhiviya Prabaa Muthuirulandi Sethuvel, V. Balaji, Nabanita Giri\nKeyw

**SYSTEM INSTRUCTIONS**

You are an expert data extraction, database schema and normalization assistant specialized in systematic reviews.
Your core task is to extract highly accurate, to-the-point variable values from the provided research papers.

RULES:
* Always execute internal reasoning step-by-step before arriving at a conclusion.
* Rely strictly on the facts explicitly reported in the document. Do not guess, or hallucinate.

<document>
# Characterizations of novel broad-spectrum lytic bacteriophages Sfin-2 and Sfin-6 infecting MDR Shigella spp. with their application on raw chicken to reduce the Shigella load

DOI: https://doi.org/10.3389/fmicb.2023.1240570
PMC: 10716491
PMID: 38094623
Published: Volume 14, (2023)
Authors: S. K. Tousif Ahamed, Srijana Rai, Chiranjib Guin, Rameez Moidu Jameela, Somasri Dam, Dhiviya Prabaa Muthuirulandi Sethuvel, V. Balaji, Nabanita Giri
Keywords: bacteriophage Shigella spp. phage therapy genome sequencing large terminase

The evidence and prevalence of multidrug-resistant (MDR) Shigella spp. poses a serious global threat to public health and the economy. Food- or water-borne MDR Shigella spp. demands an alternate strategy to counteract this threat. In this regard, phage therapy has garnered great interest from medical practitioners and researchers as a potential way to combat MDR pathogens. In this observation, we isolated Shigella phages from environmental water samples and tested against various clinically isolated MDR Shigella spp. In this study, we have defined the isolation and detailed physical and genomic characterizations of two phages Sfin-2 and Sfin-6 from environmental water samples. The phages exhibited potent lytic activity against Shigella flexneri, Shigella dysenteriae, and Shigella sonnei. They showed absorption within 5–10 min, a burst size ranging from ~74 to 265 PFU/cell, and a latent period of 5–20 min. The phages were stable at a broad pH range and survived an hour at 50°C. The purified phages Sfin-2 and Sfin-6 belong to the Siphoviridae family with an isometric head (64.90 ± 2.04 nm and 62.42 ± 4.04 nm, respectively) and a non-contractile tail (145 ± 8.5 nm and 148.47 ± 14.5 nm, respectively). The in silico analysis concluded that the size of the genomic DNA of the Sfin-2 phage is 50,390 bp with a GC content of 44.90%, while the genome size of the Sfin-6 phage is 50,523 bp with a GC content of 48.30%. A total of 85 and 83 putative open reading frames (ORFs) were predicted in the Sfin-2 and Sfin-6 phages, respectively. Furthermore, a comparative genomic and phylogenetic analysis revealed that both phages represented different isolates and novel members of the T1-like phages. Sfin-2 and Sfin-6 phages, either individually or in a cocktail form, showed a significant reduction in the viable Shigella count on raw chicken samples after 72 h of incubation. Therefore, these results indicate that these phages might have a potential role in therapeutic approaches designed for shigellosis patients as well as in the biological control of MDR Shigella spp. in the poultry or food industry during the course of meat storage.

## 1 Introduction

Shigellosis or bacillary dysentery is an acute inflammatory diarrheal disease in most of the developing countries affecting nearly 165 million people each year (WHO,). Though the number of reported deaths has decreased, shigellosis still causes high morbidity and mortality, particularly among children and young adults (Sur et al.,). The genus Shigella having four pathogenic serogroups (Shigella dysenteriae, Shigella flexneri, Shigella boydii, and Shigella sonnei) is mainly associated with Shigellosis (Kotloff et al.,; Yang et al.,). The main mode of transmission is via the fecal-oral route due to the intake of contaminated food and water (Baird-Parker,; Shahin et al.,; Pakbin et al.,). The World Health Organization (WHO) recommends antibiotics for the treatment of Shigellosis; nonetheless, the extensive use of antibiotics can lead to the development of multidrug-resistant (MDR) Shigella species (Sivapalasingam et al.,; von Seidlein et al.,; Muthuirulandi Sethuvel et al.,; Puzari et al.,). Although, recently, there have been some antibiotics suggested for the treatment, including ciprofoxacin [a fuoroquinolone (FQ)], pivmecillinam, azithromycin, and ceftriaxone (a third-generation cephalosporin) (Nandy et al.,; Tariq et al.,; Azmi et al.,), drug-tolerant persister S. flexneri and FQ-resistant Shigella spp. have still been identified in many Asian countries, including India (Taneja and Mewara,; Puzari et al.,; Sethuvel et al.,). Hence, the repetitive transition in the antimicrobial resistance behavior of Shigella hinders the development of standard drugs against shigellosis. The potential ability of these bacteria to gain and disperse exogenous genes through mobile genetic elements, such as plasmids, transposons, insertion sequences, and genomic islands, is mainly responsible for the emergence of their multidrug-resistant strains (Muthuirulandi Sethuvel et al.,; Ranjbar and Farahani,).

Bacteriophages are specific viruses that have the capability to infect and kill their target bacterial cells (Ayariga et al.,; Li et al.,; Ayariga Joseph et al.,; Gildea et al.,; Ibrahim et al.,). The characteristics of bacteriophages, such as ubiquitous nature, host specificity, safety, antimicrobial activity, and surface decontamination ability, make them a suitable agent for therapeutic purposes (Peng et al.,). Currently, antibiotic resistance is a difficult problem to overcome, and due to the host specificity, phages are not ideal for broad-spectrum use; however, a combination of different bacteriophages, known as phage cocktail, can be an ideal means to combat antibiotic-resistant bacterial strains, since the bacterial cocktail increases the host range of the phages (Lin et al.,).

There have been recent reports of several bacteriophages against Shigella spp. The lytic Shigella phages vB_SflS-ISF001, vB_SsoS-ISF002, and pSf-1 infect both S. flexneri and S. sonnei (Jun et al.,; Shahin et al.,). The lytic phage Sfk20 infects S. flexneri serotypes1b, 2a, 3a, S. sonnei, and S. dysenteriae 1 but is ineffective against S. flexneri serotypes 4, 6, and S. boydii (Mallick et al.,). The novel lytic phage Sfin-1 infects MDR S. flexneri, S. dysenteriae, and S. sonnei along with Escherichia coli C (Ahamed et al.,). The microviridae phage SGF3 has been reported to infect S. flexneri (Lu et al.,). Moreover, a number of phages that are involved in the serotype conversion of S. flexneri have been discovered, which includes SfII, Sf6, SfV, and SfX (Allison and Verma,).

In addition to water, food can also serve as a possible indirect mode of transmission of Shigella. There are reports of the isolation of Shigella from different foods, including fresh vegetables, cooked chicken meat, salads, fruits, and dairy products, ultimately leading to Shigellosis outbreaks (Shahin and Bouzari,; Pakbin et al.,). Globally, foodborne Shigella is estimated to cause 1–3 million disability-adjusted life years (DALYs) (Havelaar et al.,). Moreover, a recent analysis of Shigella isolates from more than 1,600 food samples, such as seafoods, fresh vegetables, and meats, revealed that 89% of the isolated Shigella strains were multidrug-resistant (Marami et al.,; Pakbin et al.,). Hence, effective measures are necessary to reduce Shigella-associated food-borne outbreaks and prevent the spread of resistant bacteria. One potential solution to this issue is the use of bacteriophages. In addition, using a mixture of several different phages, i.e., phage cocktail, provides a highly collaborative effect for antibacterial strength and a broad host range compared to using a single phage (Chan et al.,; Costa et al.,; Shahin et al.,).

Thus, the current study reports the isolation and detailed physical and genomic characterizations of two novel lytic bacteriophages Sfin-2 and Sfin-6. In addition, the efficacy of the novel bacteriophage cocktail consisting of these two Shigella phages was investigated based on their ability to reduce Shigella loads on raw chicken ready-to-eat meat.

## 2 Materials and methods

### 2.1 Bacterial strains and multidrug resistance test

The study analyzed 50 MDR clinical isolates of S. flexneri, S. dysenteriae, S. sonnei, S. boydii, and Salmonella enterica serovar Typhi, as well as various strains of E. coli such as AG100, K12, XL1 Blue, and E. coli C. The stool samples of patients were collected at the Bacteriology Division of National Institute of Cholera and Enteric Diseases (NICED), Kolkata and Christian Medical College (CMC), Vellore, India to obtain all Shigella and Salmonella strains, which have been reported earlier (Table 1) (Muthuirulandi Sethuvel et al.,; Ahamed et al.,; Sethuvel et al.,). For the purpose of conducting various experiments, the Luria broth (LB) with or without antibiotics was used for growing bacterial strains at 37°C. Then, the growth of these strains was checked by measuring the absorbance at 600 nm.

Table T1: Host specificity test for several clinically isolated MDR strains to Sfin-2 and Sfin-6 phages isolated from the water samples of Ganga river in Kolkata, West Bengal, India.

Sl. No.,Strain ID,Bacterial isolates with different serotypes,Antimicrobial resistance profile by disc diffusion method,Lysis by Sfin-2,Lysis by Sfin-6
1.,BCH5722,Shigella flexneri 2a (1A),ACTQNaCipNorOfx,+,+
2.,BCH4025,Shigella flexneri 2a (2A),ACQ,+,+
3.,BCH3651,Shigella flexneri 2a (3A),ACTQ,+,+
4,BCH3557,Shigella flexneri 2a (4A),CTQNa,+,+
5,BCH7151,Shigella flexneri 2a (5A),ACTQNaCipNorOfx,+,+
6,CMCFC2181,Shigella flexneri (1),AQCipCefSxt,+,+
7,BCH5762,Shigella dysenteriae 1(1A),ACTQNaCipNor,+,+
8,BCH5848,Shigella dysenteriae 1(2A),ACTQNaCipN,+,+
9,BCH5859,Shigella dysenteriae 1(3A),ACTQNaCipNorOfxAzm,+,+
10,BCH5912,Shigella dysenteriae 1(4A),ACTQNaCipNorOfx,+,+
11,BCH5946,Shigella dysenteriae 1(5A),ACTQNaCipNorOfxCef,+,+
12,CMCFC2358,Shigella dysenteriae (1),AQCipCef,+,+
13,BCH7084,Shigella sonnei (1),TQNa,+,+
14,BCH7264,Shigella sonnei (2),TQNa,+,+
15,CMCFC87,Shigella sonnei,AQNaCipCefSxt,+,+
16,CMCFC1799,Shigella sonnei,AQNaCipCefSxt,+,+
17,BCH3143,Shigella boydii (1),TQNa,–,–
18,BCH4324,Shigella boydii (2),TQNa,–,–
19,CMCFC2293,Shigella boydii,AQNaCipCefSxt,–,–
20,BCR62,Salmonella enterica serovar Typhi (1),NaCipNorAzm,–,–
21,BCR43,Salmonella enterica serovar Typhi (2),NaAzm,–,–
22,,Escherichia coli K12,,–,–
23,,Escherichia coli C,,–,–
24,,Escherichia coli AG100,,–,–
25,,XL1 Blue,,–,–

BCH, Bidhannagar Children Hospital; BCR, Bidhan Chandra Roy Hospital; CMC, Christian Medical College, Vellore; A, ampicillin; C, chloramphenicol; T, tetracycline; Q, cotrimoxazole; Na, nalidixic acid; Cip, ciprofloxacin; Nor, norfloxacin; Ofx, ofloxacin; Azm, azithromycin; Cef, cefixime; Sxt, sulfamethxazol.

### 2.2 Isolation, amplification, and purification of bacteriophages

Environmental water samples were collected from the Ganga river, near Barrackpore, North 24 Parganas district and Sreerampore, Hoogly district, which is ~25 km from Kolkata, West Bengal, India. The collected water samples were then filtered through filter paper (Whatman 1) to remove the particulate matter. The log phase S. flexneri 2a culture was added to the water sample with 10% (w/v) peptone following the incubation at 37°C for 24 h with shaking. To remove the bacterial debris, 1% (w/v) chloroform was mixed with the culture and then shaken properly. Furthermore, after the centrifugation of the mixture, the supernatant was collected and filtered through a 0.22-μm pore membrane (Millipore, USA). A volume of 10 μl of the filtrate was inoculated as a spot on a Shigella spp. plate, with subsequent formation of a clear zone around the spot indicating the presence of the bacteriophage against S. flexneri 2a. In addition, the other Shigella spp. serotypes were also included in the study.

The water samples were then used for plaque assay; 200 μl Shigella culture (OD600 = 0.3) and 100 μl filtrate were mixed together with 3.5 ml soft agar (0.9%), and finally, LB hard agar plate was used for plating. After the incubation of the plate at 37°C for 24 h, clear distinct plaques developed on the plate, which was then transferred to a separate Shigella plate. An individual plaque was shifted into a 500-μl phage dilution medium (0.85% sodium chloride and 0.1% tryptone). An additional round of plaque assay was done using the above suspended phage solution. In this way, each plaque was transferred three times for the purification of the bacteriophage.

Further, the dilutions and assaying of the phage were done to obtain a confluent lysis plate. The scrapping of the layer of soft agar was carried out and dissolved in a cold phage dilution medium (0.85% sodium chloride, 0.1% tryptone), which was retained on ice for 24 h. The supernatant was then collected after centrifugation at 5,000 × g, and the phage lysate obtained was pelleted at 68,000 g for 2 h at 4°C in an ultracentrifuge, which resulted in a higher phage titer value. Moreover, cesium chloride (CsCl) density gradient centrifugation was performed (ρ = 1.3, 1.5, 1.7 g/ml) at 100,000 g for 3 h at 4°C to obtain increased purification. The phage band captured between 1.7 and 1.5g/ml was gathered and then dialyzed against Tris-HCl magnesium sulfate (TM) buffer (50 mM Tris-Cl, pH 8.0 with 10 mM MgSO4). Finally, the phage was stored at 4°C.

### 2.3 Host range determination

The different strains of Shigella, Salmonella, and E. coli were used for determining the host range of isolated phages (Table 1). After growing them through the night in nutrient broth at 37°C, 200 μl of the bacterial cell culture was mixed with 3.5 ml of the molten soft agar (0.7% w/v) and overspread onto the surface of solid basal LB agar (1.5% w/v). A suspension phage of 10 μl (about 1.0 × 1010 PFU/ml) was used for spotting onto the bacterial lawn, which was then incubated overnight at 37°C. Clear lysis of the spot where the phage suspension was inoculated indicated the sensitivity of the bacteria. Each test was repeated three times. There were two categories of spots according to the degree of clarity: clear (+) and no reaction (–).

### 2.4 Thermal and pH stability

The thermal stability testing was performed using 1 ml of phage particles (~16 × 1013 pfu/ml for Sfin-2 and 15 × 1015 pfu/ml for Sfin-6), which were incubated at 4, 37, 50, 60, 70, 80, and 90°C, with aliquots (100 μl) taken for each temperature after 5, 15, 40, and 60 min and titered by the double-layered plaque assay against Shigella spp. Similarly, the pH stability testing was performed on phage particles (about 16 × 1012 pfu/ml) that were placed in 1 ml of TM buffer at different pH ranges between 2 and 12 (modified using HCl or NaOH for acidic or alkaline range, respectively) for 1 h at 37°C. The aliquots (100 μl) from each pH were then titered by the double-layered plaque assay against Shigella spp. (Wei et al.,).

### 2.5 Transmission electron microscopy

Ultrapure phages obtained from CsCl purification were used for electron microscopic imaging. The imaging was done at the Electron Microscopy Laboratory, University of Burdwan, West Bengal. The bacteriophage suspension (~1 × 1022 pfu/ml) was transferred onto the grid using a Gilson pipette and negatively stained with a 2% (w/v) uranyl-acetate solution. Then, it was examined under a JEOL JEM-1400Plus transmission electron microscope with an operating voltage of 200 kV.

### 2.6 Lytic activity of Sfin-2 and Sfin-6

According to the CLSI guidelines, the isolated Shigella strains were extensively drug-resistant (CLSI,). The method described by Wang et al. with some modifications was used for determining the bacteriolytic activity of phages. In the presence of various antibiotics, such as ampicillin (32 μg/ml), chloramphenicol (32 μg/ml), tetracycline (16 μg/ml), cotrimoxazole (25 μg/ml), nalidixic acid (32 μg/ml), ciprofloxacin (4 μg/ml), norfloxacin (16 μg/ml), and ofloxacin (8 μg/ml), the cells of S. flexneri 2a (strain IDBCH5722, Table 1) and S. dysenteriae 1 (strain IDBCH5762, Table 1) were grown. Similarly, in the presence of tetracycline (16 μg/ml), cotrimoxazole (25 μg/ml), and nalidixic acid (32 μg/ml), S. sonnei (strain IDBCH7084, Table 1) was grown. After centrifugation, 20 ml of cultures (OD600 = 0.3) were resuspended in 1 ml of freshly prepared LB. Furthermore, after adding phages at different multiplicity of infection (MOI) of 0.1, 0.01, and 0.001, they were allowed to adsorp for 5 min (S. flexneri 2a and S. dysenteriae 1) or 10 min (S. sonnei 1) at 37°C. Thereafter, the individual suspension was transferred to 20 ml of freshly made LB. At specific time intervals of 5 h duration, aliquots were taken and the bacterial cell count was recorded using the spread plate technique. The bacterial cultures inoculated only with phage dilution medium and respective antibiotics were used as the negative control.

### 2.7 One-step growth curve

A one-step growth curve experiments was executed by a procedure stated by Malek et al. with an alteration. Concisely, Shigella spp. (S. flexneri 2a, S. dysenteriae 1, and S. sonnei 1) were cultured in LB medium at 37°C with respective antibiotics. After centrifuging 20 ml of Shigella culture (OD600 = 0.3) at 5,000 × g at 4°C for 10 min, the resulting pellet was resublimed in 1 ml of fresh LB. Then, the phage particles at an MOI of 0.01 were mixed with Shigella culture. Thereafter, the suspension was incubated for enhanced adsorption (5 min for S. flexneri 2a and S. dysenteriae 1, 7 min for S. sonnei 1) at 37°C pursued by 104-fold of dilution with 10 ml as final volume. Subsequently, during the incubation process at 37°C, 100 μl of aliquots were taken at different time intervals up to 100 min. These samples were then mixed with 200 μl of Shigella culture, and a double-layered agar plate assay to determine the phage titration was performed. The above experiments were carried out three times for each Shigella spp. The determination of the burst size was calculated as a ratio of the average bacteriophage particles produced after the burst and the average number of phage particles adsorbed.

### 2.8 Genome sequencing and analysis

The phage samples were allowed for ultra-purification just before DNA extraction as described. A sterile 2 ml centrifuge tube (Tarsons, India) was filled with 450 μl of phage lysate. After adding 1 μl of DNase I (2,000 units/ml, NEB, USA) and 5 μl of RNaseA (10 mg/ml, Thermo Scientific, USA) to the solution, it was incubated at 37°C for 1 h. Each centrifuge tube was treated with 5 mM EDTA and then incubated at 78–80°C for 20 min to denature DNase I. Then, 250 μg of Proteinase K (SRL, Mumbai, India) was added with incubation for 2 h at 55°C. After the primary treatment of the phage sample, the genomic DNA was isolated using the phage DNA isolation kit (Norgen “Canada”) as per the manufacturer's instruction with modifications (Berg et al.,).

The kit ION Xpress (S5-00205) version 5.0.4. was utilized for accomplishing whole genome sequencing of phages. The quality of the sequence data was checked using PRINSEQ, and the reads were quality-trimmed/filtered. The filtered sequence was converted into a single contig using SPAdes 3.8.0 (Bankevich et al.,). Rapid Annotation Subsystem Technology (RAST) was used for the accomplishment of genome annotation (Aziz et al.,). The resulting nucleotide sequence of the phage genome was submitted at GenBank under accession numbers MK972831 (Sfin-2) and MN393473 (Sfin-6), respectively. By using the BLASTp program and conserved domain search (http://www.ncbi.nlm.nih.gov/), the function of the proteins encoded by various coding sequences (CDSs) was speculated (Table 2). The possible origin of replication was predicted by GeneSkew program (http://genskew.csb.univie.ac.at/). The Neural Network Promoter Prediction tool of the Berkeley Drosophila Genome Project was used to predict putative promoter regions (minimum promoter score: 0.9, http://www.fritfly.org/seq_tools/promoter.html). The ARNOLD terminator finding program was used for determining Rho-independent transcription terminators (Lesnik et al.,). The tRNA scan-SE search program (http://lowelab.ucsc.edu/tRNAscan-SE/) was used for identifying putative tRNAs, if any of them was present (Lowe and Chan,). The Mauve procedure was conducted for whole genome comparisons (http://asap.ahabs.wisc.edu/mauve/).

Table T2: Characteristics of the protein coding sequences of phage Sfin-2 and Sfin-6 according to the homology to protein database.

Predicted functional CDSs,Best blastp match and identity (%) and protein family,CDS,Start,Stop,Length (bp),CDS,Start,Stop,Length (bp)
,,Sfin-2,Sfin-6
Tail fiber protein,"Sfin-1, 100% pfam09327COG4733",1,"3,807",358,"3,450",8,"10,302","6,853","3,450"
Tail fiber,Sfin-1100%,5,"6,425","6,072",354,2,"1,050","2,834","1,784"
Tail fiber,Escherichia phage vB_EcoS_Chao,10,"10,756","10,088",669,12,"12,920","12,567",354
Tail fiber,Sfin-1 98%,78,"44,946","45,809",864,16,"17,252","16,584",669
Tail assembly protein,"Sfin-1, 100 % cl01945",2,"4,483","3,884",600,9,"10,978","10,379",600
Tail assembly protein,"Sfin-1, 100 % cd08073",3,"5,214","4,480",735,10,"11,709","10,975",735
Minor tail protein,"phi2457T, 100 % cl01908",4,"5,993","5,211",783,11,"12,488","11,706",783
Minor tail protein,"Sfin-1, 100 % cl01940",79,"45,923","46,729",807,,,,
Tail length tape-measure protein,"phi2457T, 100 % COG4942",6,"7,871","6,492","1,380",13,"15,795","12,988","2,808"
Tail length tape-measure protein,"phi2457T, 100 % pfam06791",7,"9,298","7,916","1,383",,,,
Capsid and scaffold protein,"phi2457T, 100 % COG2369",19,"16,224","15,112","1,113",25,"22,721","21,609","1,113"
Minor capsid protein,"Shigella phage phi2457T, 100%",20,"16,988","16,227",762,26,"23,485","22,724",762
Terminase large subunit,"Sfin-1, 100 % COG5410",22,"19,886","18,318","1,569",28,"26,383","24,815","1,569"
Terminase small subunit,"Shfl1, 100 % pfam16677",23,"20,450","19,926",525,29,"26,947","26,423",525
"3′-phosphatase, 5′-polynucleotide kinase","Sfin-3, 97%",32,"23,523","22,990",534,38,"30,021","29,488",534
Holin protein,"Echerichia phageADB-2, 100%",65,"37,499","37,284",216,72,"43,999","43,784",216
DNA adenine methyltransferase,"ADB-2, 100 % cl05442",72,"40,805","40,092",714,79,"47,306","46,593",714
DNA helicase,"Sfin-1, 100 % COG1061",74,"43,076","41,286","1,791",81,"49,576","47,786","1,791"
DNA helicase,"Escherichia phage ADB-2, 95% cl28899",75,"43,303","43,067",237,82,"49,803","49,567",237
DNA primase,"VbEcoS SA12KD, 99% smart00778",77,"43,925","44,845",921,1,30,992,963
Recombinase,"Escherichia phage vB_EcoS_SA30RD, 99% pfam04404",81,"47,865","47,218",648,4,"3,970","3,323",648
Exonuclease,"Sfin-1, 100% cl00641",82,"49,004","47,940","1,065",5,"5,109","4,045","1,065"

### 2.9 Genome end determination of isolated phages

A comparative analysis of the phylogenetic relationships between amino acid sequences of phage terminase large subunit and those of the other phages of a familiar packaging system can be performed for recognizing the procedure of phage packaging and determining the bacteriophage genome ends (Amarillas et al.,). Hence, the recreation of the phylogenetic tree was done using the phages with the amino acid sequences of the large terminase. In addition, the relationships between Sfin-2 and Sfin-6 phages and the other phages were analyzed. For accomplishing the phylogenetic analysis, the predicted amino acid sequences of the large terminase subunit genes of the phages were retrieved from National Center for Biotechnology Information (NCBI). In this study, molecularly analyzed bacteriophages are implicated containing well-characterized dsDNA, which has different types of packaging strategies that are dependent on terminase actions (headful, 5′-extended cos ends, 3′-extended cos ends, and direct terminal repeats). ClustalW in MEGAX with default parameters were used for aligning all the sequences. The neighbor-joining method was used to construct a phylogenetic tree, and phylogenies were determined by the bootstrap value of 1,000 replicates in MEGA X.0 version (Filipski et al.,). Furthermore, the genome ends were recognized as shown by Amarillas and Leon-Felix (Amarillas et al.,). Approximately 1 μg bacteriophage DNA was digested with separate restriction enzymes (BglII, MluI) as per the manufacturer's guidelines (NEB, USA) for identifying the presence of terminally redundant genome ends that were circularly permutated. The digests produced were then heated to 80°C for 15 min followed by cooling quickly in ice or slowly at ambient temperature. Then, the digests were loaded and run on agarose gel (0.8% w/v) in TAE electrophoresis buffer after which the gel was stained with ethidium bromide (EtBr) and visualized with UV illumination. Lastly, as a DNA molecular weight marker, GeneRuler 1 kb Plus DNA Ladder (Thermo Fisher Scientific, USA) was used.

### 2.10 Characterization of the phage receptor

To determine the receptor features of Sfin-2 and Sfin-6 for phage host interaction, the following experiments were performed as described earlier with certain alterations (Kiljunen et al.,). To determine the proteinase K effect on the adsorption of phages, S. flexneri 2a (OD600 = 0.3) was used. The host was subjected to proteinase K treatment (250 mg/ml, SRL, Mumbai, India) for 2 h at 55°C and was left for adsorption analysis at an MOI of 0.0001. Furthermore, S. flexneri 2a cells were centrifuged at 5,000 × g for 5 min to determine the inhibitory action of periodate on the phage–host interaction. The pellets so obtained were dissolved into 50 mM sodium acetate (pH 5.2) solution in the presence or absence of 200 mM NaIO4 and then incubated for 2 h in the dark. An adsorption assay was carried out with the washed cells following the incubation. Again, for Sfin-2, the S. flexneri 2a cell was primarily treated with proteinase K and allowed for a secondary treatment with periodate. Moreover, without proteinase K and sodium acetate, a control experiment was also performed to confirm that the probable effect is not the result of sodium acetate and host cell incubation at 55°C. For both of these assays, as a non-absorbing control, LB medium was used. In the control supernatant, the phage titer value was adjusted to 100%.

### 2.11 Efficacy of the isolated phages to reduce the S. flexneri 2a load on raw chicken samples by a single phage and cocktail phages

In the area where the present study was carried out, chicken is considered as a primary meat source among the meat-based food, thereby increasing the risk of Shigella spp contamination. Raw chicken was used in this experiment (Shahin and Bouzari,). The chicken was collected from a local shop and sliced aseptically under a biosafety cabinet. The pieces were then placed on sterile petridishes and stored at 4°C until further use. Shigella flexneri 2a was grown in antibiotics containing LB broth at 37°C. Aseptically S. flexneri 2a cells (±109 cfu) were carefully spread on the surface of the chicken pieces. The phage suspension of a single or cocktail of the two phages were put on to the surface of the inoculated chicken piece at a MOI of 0.1, followed by adsorption at room temperature for 10 min. Phage cocktail was prepared by adding an equal ratio of each phage. As a control, only phage suspension medium was used. After that, the treated and control samples were incubated at 4°C up to 96 h (Zhang et al.,). The number of viable S. flexneri 2a cells and the number of phages were measured at 0, 2, 24, 48, 72, and 96 h.

At each sampling time, the pieces of chicken were transferred to a sterile tube containing 5 ml of sodium magnesium (SM) buffer solution or 0.85% NaCl. Then, the samples were shaken at ambient temperature for half an hour. In order to harvest, the suspensions after transfer were centrifuged at 5,000 g for 10 min at room temperature. For phage-treated samples, the supernatant was collected in another microcentrifuge tube to ascertain the number of phages. In case of only host control, the pellet was washed thrice and resuspended in an equal volume of 0.85% NaCl solution. The bacterial cells were measured on HEA or XLD agar by the spread plate method, and the phage number was measured by plaque assay as mentioned previously.

### 2.12 Statistical analysis

To test the thermal stability, the titer value difference taken between 0 and 60 min were estimated for individual temperature. Student's t-test was applied for comparing the difference in the titer value for individual temperature to 4°C. To evaluate the details of bactericidal activity, two-way ANOVA test was performed. To analyze the phage receptor on the host cells, student's t-test was performed. To perform all statistical analysis, software GraphPad Prism 7.0 was used.

## 3 Results and discussion

### 3.1 Isolation of bacteriophages

The water samples from River Ganga were collected from different regions in and around Kolkata, and Shigella-specific phages were determined by the methods as described in Section 2. Two phages named Sfin-2 and Sfin-6 were isolated from the waters of the River Ganga that could proliferate in various strains of clinically isolated MDR Shigella spp., and they formed clear plaques of size ranging from 1.3 to 1.9 mm in diameter with well-defined boundaries in the bacterial lawn after overnight incubation at 37°C (Figures 1A, D). The absence of more than one gene of specific phage proteins such as tail tape measure protein and large terminase subunit suggests the presence of a single type of phage in the sample.

Figure F1: Shigella spp.-specific phages Sfin-2 and Sfin-6. (A, D) Plaques of Sfin-2 and Sfin-6 in the lawn of Shigella spp. Ultra-purified phages were negatively stained and examined under electron microscope as described in Section 2. (B, C, E, F) The electronmicrograph broad view of the phages in 100 and 200 nm scales.

### 3.2 Phage morphology

The morphology of purified Sfin-2 and Sfin-6 phages were observed using transmission electron microscopy (TEM), which revealed that Sfin-2 and Sfin-6 phages had an isometric head (64.90 ± 2.04 nm and 62.42 ± 4.04 nm, respectively) and a non-contractile tail (145 ± 8.5 nm and 148.47 ± 14.5 nm, respectively) anchored with a basal tuft (Figures 1B, C, E, F). The mature phage lacks a neck, base plate, spikes, or fiber. The structure of the phages according to the guidelines of the International Committee on Taxonomy of Viruses (ICTV) suggested that both of them belong to the family Siphoviridae and grouped into Caudovirales (Fauquet and Fargette,).

The vast majority (over 95%) of the reported phages belong to the order Caudovirales, which are tailed phages. According to the Ackermann, ~60% of the phages are classified under the family Siphoviridae, which have flexible and long tails.

### 3.3 Phage host range

Lytic spectrum of Sfin-2 and Sfin-6 phages were determined by spot test of pure phages on the lawn of different clinically isolated S. flexneri, S. dysenteriae, S. boydii, and S. sonnei with other enteropathogens such as Salmonella typhi and various E. coli strains, including XL1 Blue, AG100, K12, and E. coli C. The Shigella strains used in this study were resistant to various antibiotics such as amoxicillin, tetracycline, chloramphenicol, norfloxacin, ciprofloxacin, nalidixic acid, ofloxacin, cotrimoxazole, and azithromycin, which are frequently used for therapeutic purposes (Amezquita-Lopez et al.,) (Table 1). Spot tests revealed that both the phage suspensions, Sfin-2 and Sfin-6, produced clear zones of inhibition against various serotypes of S. flexneri, S. dysenteriae, and S. sonnei but did not show activity against other bacterial species. This phenomenon clearly indicated that phages are polyvalent in nature.

While phages are usually very much specific, infecting only one species of bacteria, there has been a report of some polyvalent phages (Hamdi et al.,; Ahamed et al.,). The ability to lyse multiple Shigella strains highlighted that these phages could be explored for phage therapies against shigellosis. The wide host range of both the phages determined that the CDSs that encode host specific protein and tail component would be valuable. The main mode of transmission of Shigella spp. to humans is through the fecal-oral route; hence, the isolation of Sfin-2 and Sfin-6 phages indicated fecal contamination of the river.

### 3.4 In vitro bacterial challenge test

In vitro bacterial challenge tests were performed using both the phages, Sfin-2 and Sfin-6, individually by adding the phage at an MOI of 0.1, 0.01, and 0.001 to mid-exponential phase cells (OD600 = 0.3) in the presence of multiple antibiotics chloramphenicol, ampicillin, tetracyclin, ciprofloxacin, cotrimoxazole, norfloxacin, and ofloxacin. For every single experiment, host strains were grown in the presence of respective antibiotics, whereas phage suspension medium was taken as control. Killing curves were generated by counting the viable colonies. For Sfin-2, the viability of bacterial cells was significantly decreased when infected with an MOI of 0.1, 0.01, and 0.001 and complete lysis occurred within 3.5 h in the case of S. flexneri 2a cells. For S. dysenteriae1, complete lysis occurred after 3.5 h of infection at an MOI of 0.1, while almost complete lysis occurred after 4.5 h at an MOI of 0.01 and 0.001. Shigella sonnei 1 cells were also significantly decreased, and complete lysis occurred after 3 h at an MOI of 0.1, while in the case of MOIs of 0.01 and 0.001, complete lysis occured after 3.5 h and 4.5 h of infections, respectively (p < 0.005; Figures 2A–C). In the case of S. flexneri 2a, complete lysis occurred after 3.5 h at an MOI of 0.1 and 4.5 h at MOIs of 0.01 and 0.001. The viability of bacterial cells were moderately decreased when S. dysenteriae 1 was infected with the phage Sfin-6 at different MOI. Complete lysis occurred within 4.5 h at an MOI of 0.1, whereas complete lysis occurred at 5 h at MOIs of 0.01 and 0.001. The viable count of S. sonnei 1 cells were also decreased at an MOI of 0.1 and complete lysis occurred within 3.5 h, while MOIs of 0.01 and 0.001 showed complete lysis after 4.5 h (p < 0.005; Figures 2D–F). Determining the mean differences between all three MOIs and control was done by the two-way ANOVA test, which showed that they are significant (p < 0.0001).

Figure F2: Bacterial challenge test of phage Sfin-2 and Sfin-6 on different clinical isolates of Shigella spp. Clinically isolated species of (A, D) Shigella flexneri 2a (B, E) Shigella dysenteriae 1 and (C, F) Shigella sonnei 1 were grown (OD600 = 0.3) in 20 ml of LB medium in the presence of several antibiotics. They were harvested by centrifugation, resuspended in 1 ml of LB medium, and infected with both phages at MOIs of 0.1, 0.01, and 0.001. After adsorption, the cultures were diluted 21-fold in LB medium and incubated for 5 h with shaking at 37°C. At different time intervals, viability of Shigella spp. was determined by the spread plate method. As the negative control, Shigella spp. were grown only in the presence of antibiotics. The two-way ANOVA indicated significant difference between control and phage-infected sets (p < 0.0001, n = 3).

The in vitro challenge tests established that the phages could be used to inactivate the MDR pathogenic strains of Shigella and, therefore, these phages could be useful as a bio control agent. The efficacy of those phages in controlling Shigella infection however has to be determined by in vivo studies. It is worth noting that a host population may resist long phage treatment, resulting in the emergence of bacterial insensitive mutants (BIMs). To combat this issue, a cocktail of phages may be used instead of a single phage (Amarillas et al.,). The use of phage cocktail with more than one phage that follows different infection mechanisms may solve this problem (Yamaki et al.,). The analysis of host cell lysis suggests that the MOI is directly dependent on cell death. The application of a higher number of phages on cells causes destabilization of its outer membrane, resulting in cell lysis. As this type of lysis are not due to the replication of phage and its release, it is called “lysis from without” (Brown and Bidle,).

### 3.5 Infectivity of Sfin-2 and Sfin-6

The thermal stability test was performed to investigate the heat-resistant properties of Sfin-2 and Sfin-6 phages. When the Sfin-2 phage was warmed at 37 or 50°C for 5 min, the activity remained unchanged. Then, the activity slowly decreased to 0.1–0.01% when incubated at 60 or 70°C for 5 min, and only 0.0001% activity was present when heated to 80 or 90°C for 5 min. In the case of Sfin-6 phage, 0.01%−0.001% activity was present when incubated at 50 or 60°C for 5 min and only 0.0001% activity was retained in each case when heated at 70, 80, or 90°C for 5 min. The thermal stability of both the phages was determined by monitoring the changes of titer at different temperatures (Figures 3A, B).

Figure F3: Stability of phage Sfin-2 and Sfin-6 in wide temperature and pH ranges. (A, C) Thermal stability of Sfin-2 and Sfin-6 phages at various temperatures. Sfin-2 (16 × 1013) and Sfin-6 (15 × 1015) phage particles were incubated at different temperatures in 1 ml of LB medium, and for each temperature, the number of infectious phage particles was determined using 100 μl aliquots from various time points by plaque assay against S. flexineri 2a. The result was plotted as mean ± SD (n = 3). (B, D) pH stability of phage Sfin-2 Sfin-6. In 1 ml of TM buffer having different pH, Sfin-2 (14 × 109) and Sfin-6 (16 × 109) phage particles were incubated at 37°C for 1 h, and the number of infectious phage particles from each sample was determined using 100 μl aliquots by plaque assay against S. flexineri 2a. The result was plotted as mean ± SD (n = 3).

The Shigella infection usually occurred in the intestine at acidic pH conditions (Gorden and Small,). Therefore, it is essential to know the pH stability of Sfin-2 and Sfin-6 for controlling Shigella spp. For both the phages, highest activity was observed after an incubation period of 1 h at pH 7.0 at 37°C. Approximately 30% or 17% recovery of the Sfin-2 phage and 5% or 12% recovery of the Sfin-6 phage was found at pH 4.0 and pH 12.0, respectively (p < 0.005; Figures 3C, D).

Although the activity of the above phages was affected by higher and lower temperature or pH levels, remarkable activity remained at wide temperature and pH ranges. Thus, the result concluded that Sfin-2 and Sfin-6 phages have moderate thermal stability and a wide pH tolerance, which suggests that these phages may be used for therapeutic purposes.

### 3.6 One-step growth curve

Lytic development of Sfin-2 and Sfin-6 phages were investigated in one-step growth curve experiments. The adsorption above 90% for both the phages were completed within ~5–20 min. The growth curve study of Sfin-2 phage showed a latent period of ~7 min with the average burst size of 105 PFU/cell against S. flexneri 2a. In the case of S. dysenteriae 1 and S. sonnei 1, latent periods were ~5 and 10 min with the average burst size of 74 and 101 PFU/cell, respectively (Figures 4A–C). Sfin-6 exhibited a latent period of ~5 and 13 min with the average burst size of 71 PFU/cell and 163 PFU/cell for S. flexneri 2a and S. dysenteriae 1, respectively, whereas against S. sonnei 1, Sfin-6 exhibited a latent period of ~13 min with the average burst size of 265 PFU/cell (Figures 4D–F).

Figure F4: One-step growth curve of phage Sfin-2 and Sfin-6. Shigella flexneri 2a, Shigella dysenteriae 1, and Shigella sonnei 1 were infected at an MOI of 0.01 at 37°C. After phage absorption, the cultures were diluted to 104-fold and incubated at 37°C, and the titers in PFU per ml from the infected cultures at different time points were determined. The result was plotted as mean ± SD (n = 3). (A, D), (B, E), and (C, F) Present one-step growth curves of Sfin-2 and Sfin-6 in S. flexneri 2a, S. dysenteriae 1, and S. sonnei 1, respectively.

### 3.7 Whole genome sequencing and synteny study of Sfin-2 and Sfin-6 phages

The genome sequencing is essential to understand the phage biology. The genome of Sfin-2 has 50,390 bp (GenBank accession number: MK972831) with 44.9% GC content. Among the 85 CDSs, 22 are rightward in orientation while others are leftward (Figure 5A) and 25 CDSs had annotated functions. The putative origin of replication and terminus location is ~201 nt and 43,001 nt, respectively, which could be predicted from the GC-skew analysis (Supplementary Figure S1A). The genome of Sfin-6 also possesses a circular genome of 50,523 bp (GenBank accession number: MN393473) with a GC content of 48.3%. Out of the 83 CDSs, 16 are rightward in orientation, while others are leftward (Figure 5B). Among them, 23 have annotated functions. The GC skew analysis suggested that the putative origin of replication and the terminus location of phage Sfin-6 is ~7,001 nt and 49,501 nt, respectively (Supplementary Figure S1B). No tRNA was found in both the genomes.

Figure F5: Genome organization and comparative genome analysis of Sfin-2 and Sfin-6. The Sfin-2 (A) and Sfin-2 (B) genome maps were schematically presented. The arrows indicate the predicted CDSs and the orientation of the transcription. Predicted molecular functions of CDS were indicated by different colors: Virion morphogenesis (green arrows), DNA metabolism and replication (red arrows), DNA packaging (violet arrows), cell lysis (gray arrows), and hypothetical proteins (blue arrows).

The whole genome BLAST analysis of Sfin-2 and Sfin-6 against the NCBI database showed that they are related to two phages, i.e., pSf-2 (GenBank accession number: KP085586) and phi2457T (GenBank accession number: MH917278). The genome of the Sfin-2 phage showed 91.89% similarity with psf-2 and 98.8% similarity with phi2457T, while the genome of the Sfin-6 phage showed 92.16% similarity with psf-2 and 99% similarity with phi2457T. The Mauve alignment of Sfin-2, Sfin-6, phi2457T, and pSf-2 resulted in one large LCB of 29,977 bp (green) and three small LCB of 5,338 bp (blue), 8,492 bp (red), and 6,478 bp (fluorescent green) indicating DNA regions that are homologous among the genomes. The gaps in the graphs indicate the non-identical region of the genome. Furthermore, the alignment of these phages showed some highly homologous regions with major rearrangements, which indicates that the phages share a common genome organization with different positions of genes (Supplementary Figure S2).

### 3.8 Module analysis

The comparative genome study of the two phages showed that genome sequence, genome size, GC contents, number of transcription terminator sequences, and CDSs are close to each other. Although gene sequences of predicted structural and functional proteins share high degree of homology, they are differently arranged and sometimes oppositely oriented. Maximum differences are present in the hypothetical proteins that are yet to be characterized. Approximately 72–75% genes of Sfin-2 and Sfin-6 are of unknown functions, and most of them have >78%−80% homology with their counterparts in pSf-2 and phi2457T genomes. The high degree of similarity among these phages may be due to complex evolutionary relationship, though they have been isolated from different geographical locations.

After annotation, the Sfin-2 and Sfin-6 proteins can be categorized into following functional groups: DNA metabolism and replication proteins; the downstream gene of Sfin-2 mostly contains DNA metabolism and replication proteins, which includes 3′-phosphatase, 5′-polynucleotidekinase/CDS33, phage-associated N-6-DNA adenine-methyl transferase/CDS73, DNA helicase/CDS75, 76, DNA primase/CDS78, phage-associated recombinase/CDS82, and phage exonuclease/CDS83, while the upstream and downstream parts of the Sfin-6 genome contains all of these proteins. The 3′-phosphatase, 5′-polynucleotide kinase belongs to the family pfam03767 that includes the C-terminal domain of the bifunctional enzyme T4 polynucleotide kinase/phosphatase PNKP. The role of The PNKP phosphatase domain is to catalyze the elimination of the 3′-phosphoryl group of DNA, RNA, and deoxynucleoside 3′-monophosphates. The enzyme N-6-DNA adenine-methyl transferase (DAM) is a member of pfam05869 which methylates GATC sequence of its own DNA to protect it from exonuclease. The counterpart of this enzyme is present in the Escherichia phage ADB-2, which shares 99% identity with Sfin-2 and Sfin-6. Both the phages have helicase coding genes that belong to the pfam04851 and are involved in ATP-dependent RNA or DNA unwinding. The primase encoded by phages belongs to pfam08273. The zinc finger motif and ATP binding region of the primase/helicase at N-terminal and C-terminal, respectively, have the origin recognition property. The ERF superfamily's pfam 04404 has the phage-associated recombinase domain that contains several single-stranded annealing proteins (SSAPs) such as Red-beta, Rad 52, ERF, and RecT, which may function as Rec-A dependent and independent DNA recombination pathways. This type of recombinase encoded by the phages promotes horizontal gene transfer by homologous recombination to accelerate the evolution by intra-phage gene shuffling. The recombinase in association with phage exonuclease takes part in the replication process from fork to nucleotide metabolism. The exonuclease encoding gene of both phages encodes an exonuclease VIII that is related to pfam12684 of the PDDEXK superfamily. Thus, 3′-phosphatase, 5′-polynucleotide kinase, phage recombinase, exonuclease are involved in DNA metabolism and recombination process of the phage genome after entering the host cells.

The sequence-based prediction of the Sfin-2 phage showed that upstream cluster genes are involved in viral head morphogenesis and tail component formation while upstream and downstream cluster genes of Sfin-6 are involved in viral head morphogenesis and tail component formation. CDS21 of Sfin-2 and CDS25, CDS26 of Sfin-6 are likely to produce phage capsid and scaffold protein belonging to Phage Mu protein F-like family which are required for viral head morphogenesis. Head and tail junction proteins, known as portal proteins, allow the phage genome into the pro head as a part of the packaging motor (Lokareddy et al.,). CDS23 and CDS24 of Sfin-2 and CDS28 and CDS29 of Sfin-6 encode phage large and small terminase subunits, which are involved in the packaging of concatameric DNA in phage capsids (Mobberley et al.,). CDS1, CDS4, CDS5, CDS10, CDS78, and CDS79 probably encode the tail component for Sfin-2, whereas CDS2 and CDS3 direct the synthesis of the protein responsible for tail assembly. CDS2, CDS8, CDS11, CDS12, and CDS16 encode the tail component for Sfin-6, whereas CDS9 and CDS10 direct the synthesis of the protein responsible for tail assembly. CDS6 and CDS7 for Sfin-2 and CDS13 for Sfin-6 encode tail tape measure protein which are the second largest genes of the phage genome. The tail length of the lambdoid phages may be hypothetically determined by the total amino acid residue of tail tape measure protein where a single amino acid is corresponding to ~0.15 nm (Katsura,). According to this hypothesis, the probable tail lengths of Sfin-2 and Sfin-6 phages are 140 and 138 nm long, respectively, which are much closed to the measured length of 145 and 148 nm, respectively.

CDS22 and CDS23 of Sfin-2 and CDS28 and CDS29 of Sfin-6 encode the large and small terminase subunit, respectively. These are mainly involved in ATP-dependent DNA packaging system.

CDS64 of Sfin-2 and CDS71 of Sfin-6 encode cell lysis protein lysin while CDS65 for Sfin-2 and CDS72 for Sfin-6 encode holins, which play an important role in host cell destruction during the burst step of the phage life cycle. After the assembly of new progeny of phages, the host cell lysed by a dual lysis system followed by a pore-forming holin protein and a cell wall degrading enzyme known as phage lysozyme or endolysin. Both lysin and holin encoding genes are located contagiously at the terminal part of Sfin-2 and Sfin-6 genomes. The lysin-coding gene encodes 162 amino acids along with phage lysozyme/endolysin belonging to the pfam00959 family found in dsDNA phages. Holin in association with other members of pfam 00959 cleaves the ß1,4-glycosidic linkage of polysaccharide present in the bacterial membrane (Ziedaite et al.,). CDS76 of Sfin-2 and CDS83 of Sfin-6 encode transcriptional regulatory cro protein that belongs to the HTH_XRE superfamily. Phages may use this protein to regulate transcriptional timing in the gene expression. Therefore, the presence of lysis genes and the absence of lysogeny-related genes in both the genomes clearly indicate that the phages are potent lytic phages.

### 3.9 Determination of genome ends

Whole genome sequencing followed by the assembly of both phages revealed that they had a double-stranded DNA genome. In tailed bacteriophages, a linear genome is expected within the channel of the portal protein where only one dsDNA can pass. Therefore, the head contains a linear genome with different types of ends. However, PCR with the primers designed at the two ends of the whole genome sequence confirmed the circular nature of the Sfin-2 and Sfin-6 phage genomes (Supplementary Table S1, Supplementary Figure S3). Consequently, two PCRs at the adjacent of the 5′ and 3′ end of the genome were taken as the positive control (data not shown).

Phage terminase is one of the most conserved protein that creates the virion end, and this enzyme is one of the most conserved phage proteins within the group. Therefore, the comparative analysis of terminase amino acid sequence of a phage results in the same clusters with others that generate similar ends. According to the phylogenetic analysis of the large terminase subunit, Sfin-2 and Sfin-6 clustered with the terminase of Shigella phages ISF002, Shfl1, psf-2, ISF001, and E. coli phage ADB-2 which belong to T1 family of phage (Figure 6). According to the cluster, it is predicted that both the genomes have direct terminal repeats with possible circular permutation. In such a circularly permutated headful packaging phage category, the site of initiation cleavage is not precise and several initiation cuts are spread on concatamers. Thus, for this reason, the chromosome length of individual virions are not precise. The abovementioned types of phages are expected to contain all the fragments of the restriction digestion of the circular phage genome as well as of undigested phage DNA along with submolarpac fragment-like P22 genome (Casjens et al.,). The pac fragments, such as phage sf6 and ES18, may not be detected for imprecise series initiation cleavage. Hence, as a result, a blur background will be observed due to variable lengths of terminal fragments.

Figure F6: Phylogenetic study of Sfin-2 and Sfin-6 phages with related phages. The phylogenetic analysis based on the large terminase subunit of known packaging mechanisms phages. The bootstrap analysis was performed with 1,000 repetitions. The terminase large subunits were compared in the MEGA 7.0 version using the neighbor-joining method.

Restriction digests of Sfin-2 and Sfin-6 phage genomes by BglII and MluI were warmed at 80°C and then cooled down slowly or rapidly, and no difference was noticed between slow- and fast-cooled sets for both the enzymes. However, instead, longer fragments were observed which indicated the absence of cohesive ends in both the phage genomes. Additionally, blur background was also observed in electrophoresis gel. For the phages that contain cohesive ends are expected to anneal and appear as a longer fragment in gel electrophoresis. This result indicates that both Sfin-2 and Sfin-6 phages are the T1-like headful packaging phage (Figure 7).

Figure F7: Enzymatic analysis of Sfin-2 and Sfin-6 genomic DNA. Phage Sfin-2 (A) and Sfin-6 (B) DNA was completely digested with BglII and MluI and the products were analyzed by 0.8% agarose gel electrophoresis. Lane M indicates the 1 kb Plus DNA Ladder. Lanes F and S indicate that the digests were heated to 80°C for 15 min and then cooled fast on ice or slow at room temperature, respectively.

### 3.10 Characterization of the host receptor

The important aspect of phage infection is the identification of host cell surface receptor for adsorption. The nature and location of the host cell receptors vary greatly depending on the phage and host (Stone et al.,). They range from peptide sequences to polysaccharide moieties. In fact, bacterial capsules or slime layer appendages may also act as the receptor of the phages (Sorensen et al.,; Bae and Cho,; Mahony and van Sinderen,; Dowah and Clokie,; Ha et al.,; Leprince and Mahillon,).

Shigella spp. belong to gram-negative bacteria and exhibit complex LPS and protein in their outer membrane structures. So, either LPS or protein or both of them may involve in phage host interaction (Cohen et al.,; Qasim et al.,). Therefore, it is very much essential to identify the actual component which serves as the receptor of the phages. Based on the strategy of Kiljunen et al., the outer membrane LPS and protein of S. flexneri 2a were degraded by periodate and proteinase K before the infection (Kiljunen et al.,; Stone et al.,). The Sfin-2 phage showed no changes in infection efficiency with or without proteinase K and periodate-treated host. In contrast, a high number of phage particles remained unabsorbed when hosts were pre-treated with proteinase K and periodates at a time. Thus, this experiment suggests that the adsorption of phage Sfin-2 phage to the host is mediated either by the outer membrane of the protein or complex LPS structure (Figure 8A). In the case of Sfin-6 phage, a high number of residual phage were present when S. flexneri 2a cells were pre-treated with periodates whereas no significant change in efficacy of infection was observed when the host cell was pre-treated with proteinase K. Therefore, this result suggests that the adsorption of Sfin-6 phage to the host is mediated by the outer membrane LPS structure but not the protein (Figure 8B).

Figure F8: Sfin-2 and Sfin-6 infections on proteinase K and periodate-treated host. The effect of proteinase K and sodium periodate with proteinase K on Sfin-2 (A). The effect of sodium periodate and sodium periodate with proteinase K on Sfin-6 (B). Shigella flexneri 2a culture (OD600 = 0.3 U) was treated with proteinase K (250 mg/ml), sodium periodate (200 mM ), and sodium periodate with proteinase K followed by infection at an MOI of 0.0001. Upon centrifugation, the phage titer in the supernatant was determined by plaque assay. Cells suspended in LB medium, cells incubated at 55°C in LB medium, and cells in acetate buffer were used as control. The results are shown as residual PFU percentages. The phage titer in the control supernatant was set to 100%. The mean ± SD of three independent experiments is indicated. To determine the significance of the differences between group means, unpaired t-tests were performed between the controls and the tests. Asterisks indicate the significance levels (ns, p > 0.05; *p ≤ 0.05; **p ≤ 0.005).

### 3.11 Inactivation of S. flexneri 2a cells with Sfin-2 and Sfin-6 by singly or cocktail of two phages in raw chicken sample

Foodborne infections are major threats to food safety in the present times. Recently, nearly two billion individuals are suffering from foodborne illnesses, resulting in 1 million deaths around the world (Kirk et al.,). Traditional food sanitation techniques can be effective in reducing the presence of pathogens in food with varying degrees. However, these methods have plenty of disadvantages, including the damage of organoleptic qualities of foods, and most importantly, chemicals used in food safety eliminate “good” microbes that are beneficial in the natural preservation of foods (Moye et al.,). Therefore, it is preferable to use bacteriophages as an alternative tool to combat the problems, as the bacteriophages are host-specific and kill their respective hosts without changing organoleptic properties of foods with low-cost large scale production, self-replicating nature, and low toxicity (Loc-Carrillo and Abedon,; Perera et al.,). The use of bacteriophages to control MDR pathogens is gaining more interest in recent times (Rogovski et al.,). Zhang et al. reduced the Shigella load on ready-to-eat spiced chicken by at least 2log10 after using Shigella-specific phages. Shahin et al. reported significant reduction of Shigella contamination in food items after the uses of Shigella-specific phages (Shahin and Bouzari,). In this study, the two polyvalent Shigella phages, Sfin-2 and Sfin-6, were used either individually or in a cocktail form to reduce the Shigella load on raw chicken samples. The result showed significant differences in the number of viable bacterial cells between the control and single phage or cocktail-treated chicken sample. No Shigella cells were found in control. The concentration of viable bacterial cells on the treated chicken sample by both single and cocktail of phages decreased by ~2log10 of the initial count. The major reduction in cell concentration occurred after 48 h of incubation, and almost complete lysis occurred after 72 h. At 96 h of incubation, the viability of cells reduced below the level of detection (Figure 9A).

Figure F9: Inactivation of Shigella flexneri 2a by the single phage or cocktail of phages on chicken. Mid-log phase culture of S. flexneri 2a was inoculated on the surface of the chicken and allow for incubation of 10 min. Afterward, a single phage or cocktail form of phages was added (MOI of 0.1) and kept at 4°C up to 96 h. As the negative control, chicken pieces were inoculated with S. flexneri 2a without any phages. At different time intervals, the viability of S. flexneri 2a was determined by the spread plate method, and the number of phages was measured by plaque assay. (A) Reduction in viable count of cells showed after 48 h of incubation. A two-way ANOVA test indicated significant difference between control and phage infected sets (p < 0.0001, n = 3). (B) The number of phages decreased initially for 2 h but after that, the number increased gradually.

The number of active phages were also measured at each time point after treatment. The number of phage decreased by ~2log10 of the initial value, 2 h after the addition of single or cocktail phages. Afterward, the number gradually increased with time in both single and cocktail of phages (Figure 9B).

## 4 Conclusion

Shigellosis is still one of the major threats in developing countries, and multidrug resistance of Shigella spp. has made the situation even worse. Therefore, to combat the situation, phages are gaining more popularity as an alternative therapeutic agent to resist pathogenic bacterial infection. Other than that, phages are also useful to treat foods infected with MDR bacterial pathogens. In the present study, we have characterized two novel thermostable and wide pH-tolerant Siphoviridae phages, Sfin-2 and Sfin-6, that have specificity and lytic properties against important enteropathogenic MDR Shigella spp. The article represents the complete physical as well as genomic characterizations of the Sfin-2 and Sfin-6 phages that include sequence analysis, genome annotations, and differences between gene rearrangements among the other closely related phages. Genome analysis is very crucial for the study and use of phages to regulate host bacterial machinery. Phylogenetic analysis confirms that Sfin-2 and Sfin-6 belong to the T1-like phage family, which may be packaged by the headful packaging method. The phage–host interaction study through specific receptor molecules suggested that the phage Sfin-2 can interact with both LPS-O antigen and protein, while Sfin-6 only interacts with the LPS-O antigen of the outer cell membrane of the host cells. Further studies of the activity of Sfin-2 and Sfin-6 phages on Shigella-infected raw chicken meat either in a single or cocktail form ensure that both the phages have the potential to reduce the number of MDR Shigella load from the meat samples.

From the present study, it can be concluded that the Sfin-2 and Sfin-6 phages can be satisfactory therapeutic agents either in a single or cocktail form, and further studies on these two phages will be helpful to apply it for the treatment of shigellosis as well as for the preservation of meat.

## Data availability statement

The datasets presented in this study can be found in online repositories. The names of the repository/repositories and accession number(s) can be found in the article/Supplementary material.

## Author contributions

SA, SR, CG, and NG conceived and designed the entire study, performed the experiments, analyzed the results, and prepared the manuscript. DM and VB supplied the clinical samples. SA, NG, and SD analyzed the phage structure. RJ helped in genome analysis. All authors wrote, read, and approved the final manuscript.

## Conflict of interest

The authors declare that the research was conducted in the absence of any commercial or financial relationships that could be construed as a potential conflict of interest.

## Publisher's note

All claims expressed in this article are solely those of the authors and do not necessarily represent those of their affiliated organizations, or those of the publisher, the editors and the reviewers. Any product that may be evaluated in this article, or claim that may be made by its manufacturer, is not guaranteed or endorsed by the publisher.

## Supplementary material

The Supplementary Material for this article can be found online at: https://www.frontiersin.org/articles/10.3389/fmicb.2023.1240570/full#supplementary-material

</document>

In [8]:
pass1_fact_extraction = (
    "### Task\n"
    "{conditions}\n\n"
    "Find the relevant value(s) for '{target_column}'.\n\n"
    "Example style reference: '{example_format}'. (do NOT use this value, only match its format/style)\n\n"
    
    "### Researcher Reading Protocol\n"
    "Simulate a human researcher by scanning the document sections in this specific priority order:\n"
    "1. Abstract / Executive Summary / Introduction: Establish high-level context and core definitions.\n"
    "2. Main Findings, Data, & Exhibits: Analyze core data summaries, tables, and primary results.\n"
    "3. Methodology, Process, & Setup: Check here for technical frameworks, procedures, or underlying data sources.\n"
    "4. Discussion, Conclusion, & Appendices: Use as a fallback for secondary mentions or supplementary details.\n\n"

    "### Thinking Guidelines\n"
    "Process your internal reasoning using the following sequential pattern:\n"
    "1. Cleaned Question: Rephrase the target query for absolute clarity.\n"
    "2. Source Quotes: Copy the verbatim text segments containing the target facts.\n"
    "3. Synthesis: Align the raw facts with the requested format.\n\n"
    
    "### Output Guidline\n" 
    " - List each final value on a separate line, prefixed with V1: <value>, V2: <value>, etc.\n"
    " - If only one value is found, prefix it with V1: <value>\n"
    " - If the entity or value is missing from the text, output exactly: V1: NOT AVAILABLE\n\n"
    " - CONSTRAINT: Do NOT output markdown headers, explanations, introduction text, etc, in your final response"
)

row, column, conditions, example_val = build_conditions_and_example(tbl_extract)
pass1_prompt = format_extraction_prompt(column, conditions, example_val, pass1_fact_extraction)

chat._messages[:] = chat._messages[:2]
chat.add_user_message(pass1_prompt)

Markdown(pass1_prompt)

### Task
When the 'Phage Name' is 'Sfin-2'
and 
the 'Phage Genome Accession/Bioproject' is 'MK972831'
and 
the 'Phage Genome size (bp)' is '50,390 bp'
and 
the 'Phage GC content (%)' is '44.9'
and 
the 'Phage TEM shows structural similarity with ' is 'NOT AVAILABLE'
and 
the 'Phage TEM dimensions/Capsid morphology' is 'Isometric head (64.90 ± 2.04 nm) and non-contractile tail (145 ± 8.5 nm)'
and 
the 'Phage Taxonomy' is 'Siphoviridae, Caudovirales'
and 
the 'Phage type: Lytic/ Lysogenic/ Engineered' is 'Lytic'
and 
the 'Place of Sample collection ' is 'Ganga river, near Barrackpore, North 24 Parganas district and Sreerampore, Hoogly district, which is ~25 km from Kolkata, West Bengal, India'
and 
the 'Phage isolation Sample' is 'Ganga river, near Barrackpore, North 24 Parganas district and Sreerampore, Hoogly district, which is ~25 km from Kolkata, West Bengal, India'

Find the relevant value(s) for 'Studied Host Strain/ Propagating Bacterial Strain'.

Example style reference: 'A. hydrophila CECT 839T'. (do NOT use this value, only match its format/style)

### Researcher Reading Protocol
Simulate a human researcher by scanning the document sections in this specific priority order:
1. Abstract / Executive Summary / Introduction: Establish high-level context and core definitions.
2. Main Findings, Data, & Exhibits: Analyze core data summaries, tables, and primary results.
3. Methodology, Process, & Setup: Check here for technical frameworks, procedures, or underlying data sources.
4. Discussion, Conclusion, & Appendices: Use as a fallback for secondary mentions or supplementary details.

### Thinking Guidelines
Process your internal reasoning using the following sequential pattern:
1. Cleaned Question: Rephrase the target query for absolute clarity.
2. Source Quotes: Copy the verbatim text segments containing the target facts.
3. Synthesis: Align the raw facts with the requested format.

### Output Guidline
 - List each final value on a separate line, prefixed with V1: <value>, V2: <value>, etc.
 - If only one value is found, prefix it with V1: <value>
 - If the entity or value is missing from the text, output exactly: V1: NOT AVAILABLE

 - CONSTRAINT: Do NOT output markdown headers, explanations, introduction text, etc, in your final response

In [14]:
pass1_response = model.respond(chat)

pass1_raw = pass1_response.content
pass1_thought, pass1_text = split_llm_response(pass1_raw)
Markdown(pass1_thought + pass1_text)

1.  **Analyze the Request:** The goal is to determine how to group or split 'Shigella dysenteriae' and 'Shigella sonnei' based on the criteria for the blank fields (Primary targeted bacteria species, Plaque characteristics, Optimal MOI, Latent period, Burst size, Optimal Temperature, Optimal pH).

2.  **Examine Candidates:** 'Shigella dysenteriae' and 'Shigella sonnei'.

3.  **Check Candidate 1: Primary targeted bacteria species (not Strain):**
    *   Scan the text for mentions of these species in the context of host range/specificity.
    *   Section 1, Paragraph 2 mentions: "The genus Shigella having four pathogenic serogroups (Shigella dysenteriae, Shigella flexneri, Shigella boydii, and Shigella sonnei) is mainly associated with Shigellosis".
    *   Section 2.1 mentions "Shigella flexneri 2a, S. dysenteriae 1, and S. sonnei 1" in the context of growth curves/challenge tests.
    *   Section 3.3 mentions "S. flexneri, S. dysenteriae, and S. sonnei" in the context of host range determination.
    *   *Conclusion for Primary targeted bacteria species:* Both species are targeted.

4.  **Check Candidate 2: Phage's Plaque characteristics/Shape:**
    *   Scan Section 3.1 and 3.2 for plaque size and morphology.
    *   Section 3.1 states: "Two phages named Sfin-2 and Sfin-6 were isolated from the waters of the River Ganga that could proliferate in various strains of clinically isolated MDR Shigella spp., and they formed clear plaques of size ranging from 1.3 to 1.9 mm in diameter with well-defined boundaries in the bacterial lawn after overnight incubation at 37°C".
    *   *Conclusion for Phage's Plaque characteristics/Shape:* Both phages have the same plaque characteristics (size 1.3-1.9 mm__LM_STUDIO_INTERNAL_LSEP_SYNTHETIC_REASONING_END_f4e9a8d2c6b14d0c9e5f3a7b8c1d2e6a__

In [10]:
# pass2 V1
pass2_split_decision = (
    "### TASK\n"
    "Analyze the extracted facts against the provided document to determine if they represent a single grouped entity or distinct entities requiring individual database rows.\n\n"

    "### DECISION RULE & THINKING STEPS\n"
    "1. Check if the values are associated with distinct, individual results/measurements reported in the document.\n"
    "2. Separate Rows: If individual values have separate, corresponding metrics mapped directly to them, split them into separate lines.\n"
    "3. Single Row: If multiple values are merely listed together without individual metrics or differentiated results, group them into a single string on one line.\n\n"
    
    "### Reference Examples\n"
    "*Example A* (Grouped Mention -> SINGLE LINE):\n"
    " - Target Column: 'Tested Solvents'\n"
    " - Extracted Facts: ethanol, methanol, and acetone\n"
    " - Document Context: 'We tested solubility in ethanol, methanol, and acetone. All tests were successful.' (No individual metrics per solvent).\n"
    " - Analysis: The solvents do not have separate individual measurements reported. They must be grouped.\n"
    " - Output Values: V1: ethanol, methanol, and acetone\n\n"
    
    "*Example B* (Actively Differentiated -> MULTIPLE LINES):\n"
    " - Target Column: 'Tested Compounds'\n"
    " - Extracted Facts: Compound A, Compound B\n"
    " - Document Context: 'Compound A degraded at 150°C, while Compound B degraded at 210°C.'\n"
    " - Analysis: Distinct compounds with separate corresponding measurements. Requires separate lines.\n"
    " - Output Values:\n"
    "   V1: Compound A\n"
    "   V2: Compound B\n\n"
    
    "### Input Data\n"
    "- Target Column: '{target_column}'\n"
    "- Extracted Facts:\n"
    "{pass1_output}\n\n"
    
    "### Decision Output:\n"
    "List the finalized values under a single 'Values:' header using the format V1: <value>, V2: <value>, etc. (one entity per line). Do not add any conversational text."
)

# pass2_prompt = pass2_split_decision.format(
#     target_column=column,
#     pass1_output=pass1_text
# )

# chat._messages[:] = chat._messages[:2]
# chat.add_user_message(pass2_prompt)

# Markdown(pass2_prompt)

In [15]:
print(pass1_text)

1.  **Analyze the Request:** The goal is to determine how to group or split 'Shigella dysenteriae' and 'Shigella sonnei' based on the criteria for the blank fields (Primary targeted bacteria species, Plaque characteristics, Optimal MOI, Latent period, Burst size, Optimal Temperature, Optimal pH).

2.  **Examine Candidates:** 'Shigella dysenteriae' and 'Shigella sonnei'.

3.  **Check Candidate 1: Primary targeted bacteria species (not Strain):**
    *   Scan the text for mentions of these species in the context of host range/specificity.
    *   Section 1, Paragraph 2 mentions: "The genus Shigella having four pathogenic serogroups (Shigella dysenteriae, Shigella flexneri, Shigella boydii, and Shigella sonnei) is mainly associated with Shigellosis".
    *   Section 2.1 mentions "Shigella flexneri 2a, S. dysenteriae 1, and S. sonnei 1" in the context of growth curves/challenge tests.
    *   Section 3.3 mentions "S. flexneri, S. dysenteriae, and S. sonnei" in the context of host range det

In [13]:
def build_decision_tree_pass2_prompt(target_column, pass1_text, blank_fields):
    """
    Constructs a hyper-focused, checklist-driven Pass 2 prompt
    tailored specifically to the extracted candidates and blank columns.
    """
    import re
    
    # 1. Parse candidates from Pass 1 output (lines starting with V1:, V2:, etc.)
    candidates = []
    for line in pass1_text.split('\n'):
        match = re.match(r'^V\d+:\s*(.*)', line.strip(), re.IGNORECASE)
        if match:
            val = match.group(1).strip()
            if val and val.lower() not in ["not specified", ""]:
                candidates.append(val)
                
    # Fallback to splitting by common separators if no structured list was found
    if not candidates:
        cleaned_raw = pass1_text.strip().strip("[]'\"")
        candidates = [c.strip() for c in re.split(r',|;', cleaned_raw) if c.strip()]
        
    # If we still have less than 2 candidates, returning a simple group rule
    if len(candidates) < 2:
        candidates = [pass1_text.strip()]

    candidates_str = ", ".join([f"'{c}'" for c in candidates])
    blank_fields_str = ", ".join([f"'{f}'" for f in blank_fields])
    
    # 2. Build the step-by-step checklist dynamically based on blank fields
    checklist_lines = []
    for i, field in enumerate(blank_fields, 1):
        checklist_lines.append(
            f"{i}. Does the document report distinct, different measurements/values for each candidate ({candidates_str}) "
            f"under the field '{field}'? (Write down 'Yes' or 'No', and cite the specific values/quotes if Yes)."
        )
    checklist_str = "\n".join(checklist_lines)

    # 3. Build Case examples matching the specific candidates
    case_1_output = "\n".join([f"    V{idx}: {c}" for idx, c in enumerate(candidates, 1)])
    case_2_output = f"    V1: {', '.join(candidates)}"
    
    case_3_example = ""
    if len(candidates) >= 3:
        case_3_example = (
            f"- CASE 3 (Partial Split): If one candidate (e.g. '{candidates[0]}') has a distinct value, but the other candidates "
            f"('{candidates[1]}', '{candidates[2]}') share matching values or both have missing data:\n"
            f"  Output Format:\n"
            f"    V1: {candidates[0]}\n"
            f"    V2: {', '.join(candidates[1:])}\n\n"
        )

    # 4. Assemble the prompt
    prompt_lines = [
        "### TASK",
        f"Decide how to group or split the following candidate entities extracted for '{target_column}':",
        f"Candidates-> {candidates_str}",
        "",
        "### RELATIONAL RULE",
        f"We are checking if these candidates should occupy separate rows based on whether they have unique, differentiated values in the document for any of these upcoming blank fields:",
        f"📋 Upcoming Blank Fields: {blank_fields_str}",
        "",
        "### STEP-BY-STEP CHECKLIST FOR YOUR THOUGHTS (<|channel>thought)",
        "Analyze the research article and answer these questions inside your thought channel first:",
        checklist_str,
        "",
        "### DECISION TREE",
        "Based on your checklist answers, choose and output one of these formatting cases:",
        "",
        "- CASE 1 (Split All): If you answered YES to any question (every candidate has unique values for the empty fields):\n"
        "  Output Format:\n"
        f"{case_1_output}\n\n"
        
        "- CASE 2 (Group All): If you answered NO to all questions (candidates share the same values or have no values reported):\n"
        "  Output Format:\n"
        f"{case_2_output}\n\n"
        
        f"{case_3_example}"
        "### OUTPUT FORMAT",
        "List the finalized values under a single 'Values:' header using the chosen V1:, V2: format exactly. Do not write any conversational text or markdown outside the headers."
    ]
    
    return "\n".join(prompt_lines)


# Extract candidates and blank fields to build the dynamic prompt
current_row = tbl_extract.iloc[row]
blank_cols = [
    col for col in tbl_extract.columns 
    if col != column and (pd.isna(current_row[col]) or current_row[col] == "" or current_row[col] is None)
]

# Build the dynamic prompt
pass2_prompt = build_decision_tree_pass2_prompt(
    target_column=column,
    pass1_text=pass1_text,
    blank_fields=blank_cols
)

chat._messages[:] = chat._messages[:2]
chat.add_user_message(pass2_prompt)

Markdown(pass2_prompt)

### TASK
Decide how to group or split the following candidate entities extracted for 'Studied Host Strain/ Propagating Bacterial Strain':
Candidates-> 'Shigella dysenteriae', 'Shigella sonnei'

### RELATIONAL RULE
We are checking if these candidates should occupy separate rows based on whether they have unique, differentiated values in the document for any of these upcoming blank fields:
📋 Upcoming Blank Fields: 'Primary targeted bacteria species (not Strain)', 'Phage's Plaque characteristics/Shape', 'Optimal MOI', 'Latent period (min)', 'Burst size (phage/infected bacterium)', 'Optimal Temperature (°C)', 'Optimal pH'

### STEP-BY-STEP CHECKLIST FOR YOUR THOUGHTS (<|channel>thought)
Analyze the research article and answer these questions inside your thought channel first:
1. Does the document report distinct, different measurements/values for each candidate ('Shigella dysenteriae', 'Shigella sonnei') under the field 'Primary targeted bacteria species (not Strain)'? (Write down 'Yes' or 'No', and cite the specific values/quotes if Yes).
2. Does the document report distinct, different measurements/values for each candidate ('Shigella dysenteriae', 'Shigella sonnei') under the field 'Phage's Plaque characteristics/Shape'? (Write down 'Yes' or 'No', and cite the specific values/quotes if Yes).
3. Does the document report distinct, different measurements/values for each candidate ('Shigella dysenteriae', 'Shigella sonnei') under the field 'Optimal MOI'? (Write down 'Yes' or 'No', and cite the specific values/quotes if Yes).
4. Does the document report distinct, different measurements/values for each candidate ('Shigella dysenteriae', 'Shigella sonnei') under the field 'Latent period (min)'? (Write down 'Yes' or 'No', and cite the specific values/quotes if Yes).
5. Does the document report distinct, different measurements/values for each candidate ('Shigella dysenteriae', 'Shigella sonnei') under the field 'Burst size (phage/infected bacterium)'? (Write down 'Yes' or 'No', and cite the specific values/quotes if Yes).
6. Does the document report distinct, different measurements/values for each candidate ('Shigella dysenteriae', 'Shigella sonnei') under the field 'Optimal Temperature (°C)'? (Write down 'Yes' or 'No', and cite the specific values/quotes if Yes).
7. Does the document report distinct, different measurements/values for each candidate ('Shigella dysenteriae', 'Shigella sonnei') under the field 'Optimal pH'? (Write down 'Yes' or 'No', and cite the specific values/quotes if Yes).

### DECISION TREE
Based on your checklist answers, choose and output one of these formatting cases:

- CASE 1 (Split All): If you answered YES to any question (every candidate has unique values for the empty fields):
  Output Format:
    V1: Shigella dysenteriae
    V2: Shigella sonnei

- CASE 2 (Group All): If you answered NO to all questions (candidates share the same values or have no values reported):
  Output Format:
    V1: Shigella dysenteriae, Shigella sonnei

### OUTPUT FORMAT
List the finalized values under a single 'Values:' header using the chosen V1:, V2: format exactly. Do not write any conversational text or markdown outside the headers.

In [ ]:
# Short-circuit prompt (no benifit)
def build_decision_tree_pass2_prompt(target_column, pass1_text, blank_fields):
    """
    Constructs a hyper-focused, checklist-driven Pass 2 prompt
    with a short-circuit rule to stop thinking early if a split is confirmed.
    """
    import re
    
    # 1. Parse candidates from Pass 1 output (lines starting with V1:, V2:, etc.)
    candidates = []
    for line in pass1_text.split('\n'):
        match = re.match(r'^V\d+:\s*(.*)', line.strip(), re.IGNORECASE)
        if match:
            val = match.group(1).strip()
            if val and val.lower() not in ["not specified", ""]:
                candidates.append(val)
                
    # Fallback to splitting by common separators if no structured list was found
    if not candidates:
        cleaned_raw = pass1_text.strip().strip("[]'\"")
        candidates = [c.strip() for c in re.split(r',|;', cleaned_raw) if c.strip()]
        
    # If we still have less than 2 candidates, returning a simple group rule
    if len(candidates) < 2:
        candidates = [pass1_text.strip()]

    candidates_str = ", ".join([f"'{c}'" for c in candidates])
    blank_fields_str = ", ".join([f"'{f}'" for f in blank_fields])
    
    # 2. Build the step-by-step checklist dynamically based on blank fields
    checklist_lines = []
    for i, field in enumerate(blank_fields, 1):
        checklist_lines.append(
            f"{i}. Does the document report distinct, different measurements/values for each candidate ({candidates_str}) "
            f"under the field '{field}'? (Answer Yes/No, cite quotes if Yes)."
        )
    checklist_str = "\n".join(checklist_lines)

    # 3. Build Case examples matching the specific candidates
    case_1_output = "\n".join([f"    V{idx}: {c}" for idx, c in enumerate(candidates, 1)])
    case_2_output = f"    V1: {', '.join(candidates)}"
    
    case_3_example = ""
    if len(candidates) >= 3:
        case_3_example = (
            f"- CASE 3 (Partial Split): If one candidate (e.g. '{candidates[0]}') has a distinct value, but the other candidates "
            f"('{candidates[1]}', '{candidates[2]}') share matching values or both have missing data:\n"
            f"  Output Format:\n"
            f"    V1: {candidates[0]}\n"
            f"    V2: {', '.join(candidates[1:])}\n\n"
        )

    # 4. Assemble the prompt
    prompt_lines = [
        "### TASK",
        f"Decide how to group or split the following candidate entities extracted for '{target_column}':",
        f"👉 Candidates: {candidates_str}",
        "",
        "### RELATIONAL RULE",
        f"We are checking if these candidates should occupy separate rows based on whether they have unique, differentiated values in the document for any of these upcoming blank fields:",
        f"📋 Upcoming Blank Fields: {blank_fields_str}",
        "",
        "### STEP-BY-STEP CHECKLIST FOR YOUR THOUGHTS (<|channel>thought)",
        f"Evaluate the upcoming blank fields in order:",
        checklist_str,
        "",
        "⚡️ SHORT-CIRCUIT RULE:",
        f"As soon as you find a field where ALL candidates ({candidates_str}) have distinct, different values reported (confirming CASE 1: Split All), you MUST STOP your analysis immediately. Do not write thoughts for the remaining checklist questions. Skip directly to the output under CASE 1.",
        "",
        "### DECISION TREE",
        "Based on your checklist answers, choose and output one of these formatting cases:",
        "",
        "- CASE 1 (Split All): If you answered YES to any question (every candidate has unique values for the empty fields):\n"
        "  Output Format:\n"
        f"{case_1_output}\n\n"
        
        "- CASE 2 (Group All): If you answered NO to all questions (candidates share the same values or have no values reported):\n"
        "  Output Format:\n"
        f"{case_2_output}\n\n"
        
        f"{case_3_example}"
        "### OUTPUT FORMAT",
        "List the finalized values under a single 'Values:' header using the chosen V1:, V2: format exactly. Do not write any conversational text or markdown outside the headers."
    ]
    
    return "\n".join(prompt_lines)


# Extract candidates and blank fields to build the dynamic prompt
current_row = tbl_extract.iloc[row]
blank_cols = [
    col for col in tbl_extract.columns 
    if col != column and (pd.isna(current_row[col]) or current_row[col] == "" or current_row[col] is None)
]

# Build the dynamic prompt
pass2_prompt = build_decision_tree_pass2_prompt(
    target_column=column,
    pass1_text=pass1_text,
    blank_fields=blank_cols
)

chat._messages[:] = chat._messages[:2]
chat.add_user_message(pass2_prompt)

Markdown(pass2_prompt)

In [50]:
def build_pass2_prompt(target_column, pass1_text, blank_fields):
    return (
        "### TASK\n"
        f"* Based on the document context, You need to decide if each values: [{pass1_text}] needs separate row in the database or they bellowing to same row in database.\n"
        "### Rule\n"
        f"- Separate row: if the article gives a specific value for [{blank_fields}].\n"
        "- Same row (comma-separated): if only mentioned without those specific values.\n"
        
        "Do not extract or list the actual datapoints, justification, conversational text etc. Output ONLY this exact format\n"
        "V1: [Separate / Same]\n"
        "V2: [Separate / Same]\n"
        "V3: [Separate / Same]"
    )

# Extract candidates and blank fields to build the dynamic prompt
current_row = tbl_extract.iloc[row]
blank_cols = [
    col for col in tbl_extract.columns 
    if col != column and (pd.isna(current_row[col]) or current_row[col] == "" or current_row[col] is None)
]

# Build the dynamic prompt
pass2_prompt = build_pass2_prompt(
    target_column=column,
    pass1_text=pass1_text,
    blank_fields=blank_cols
)

chat._messages[:] = chat._messages[:1]
chat.add_user_message(pass2_prompt)

# print(chat)
Markdown(pass2_prompt)

### TASK
* Based on the document context, You need to decide if each values: [V1: Sh. flexneri 2a, V2: Sh. dysenteriae 1, V3: Sh. sonnei 1] needs separate row in the database or they bellowing to same row in database.
### Rule
- Separate row: if the article gives a specific value for [['Primary targeted bacteria species (not Strain)', "Phage's Plaque characteristics/Shape", 'Optimal MOI', 'Latent period (min)', 'Burst size (phage/infected bacterium)', 'Optimal Temperature (°C)', 'Optimal pH']].
- Same row (comma-separated): if only mentioned without those specific values.
Do not extract or list the actual datapoints, justification, conversational text etc. Output ONLY this exact format
V1: [Separate / Same]
V2: [Separate / Same]
V3: [Separate / Same]

In [51]:
pass2_response = model.respond(chat)
pass2_raw = pass2_response.content
pass2_thought, pass2_text = split_llm_response(pass2_raw)

Markdown(pass2_thought + pass2_text)

*   Task: Decide whether the three bacterial strains (Shigella flexneri 2a, Shigella dysenteriae 1, Shigella sonnei 1) need separate rows or if they belong to the same row based on specific criteria.
    *   Criterion for Separate Row: If the article gives specific values for: [Primary targeted bacteria species (not Strain), Phage's Plaque characteristics/Shape, Optimal MOI, Latent period (min), Burst size (phage/infected bacterium), Optimal Temperature (°C), Optimal pH].
    *   Criterion for Same Row: If only mentioned without those specific values.
*   Input: The provided document.

*   Analysis Strategy: Scan the document for mentions of Sfin-2 and Sfin-6 and their interaction/activity with S. flexneri 2a (V1), S. dysenteriae 1 (V2), and S. sonnei 1 (V3), specifically looking for the seven required parameters.

    *   **Sfin-2 and Sfin-6 general activity:**
        *   "The phages exhibited potent lytic activity against Shigella flexneri, Shigella dysenteriae, and Shigella sonnei. They showed absorption within 5–10 min, a burst size ranging from ~74 to 265 PFU/cell, and a latent period of 5–20 min." (General characteristics of the phages, not specific to one host).
        *   "Sfin-2 and Sfin-6 phages, either individually or in a cocktail form, showed a significant reduction in the viable Shigella count on raw chicken samples after 72 h of incubation." (Application in food).

    *   **Host range/Specificity (Table T1, Section 2.2, 3.3, 3.4, 3.6, 3.9):**
        *   Table T1 shows both Sfin-2 and Sfin-6 lysis for multiple strains (S. flexneri 2a (1A) to S. sonnei (2), etc.). This implies they target multiple serotypes.
        *   "Spot tests revealed that both the phage suspensions, Sfin-2 and Sfin-6, produced clear zones of inhibition against various serotypes of S. flexneri, S. dysenteriae, and S. sonnei but did not show activity against other bacterial species." (Implies they are polyvalent for these three, but doesn't give specific MOI, burst size, etc., for each host individually).
        *   In vitro challenge tests (Section 3.4): Mentions S. flexneri 2a, S. dysenteriae 1, and S. sonnei 1.
            *   For Sfin-2: "complete lysis occurred within 3.5 h in the case of S. flexneri 2a cells."
            *   For Sfin-2: "complete lysis occurred after 3.5 h of infection at an MOI of 0.1, while almost complete lysis occurred after 4.5 h at an MOI of 0.01 and 0.001." (Wait, this refers to S. flexneri 2a).
            *   For Sfin-2: "complete lysis occurred after 3.5 h at an MOI of 0.1 and 4.5 h at MOIs of 0.01 and 0.001."
            *   For Sfin-6: "complete lysis occurred within 4.5 h at an MOI of 0.1, whereas complete lysis occurred at 5 h at MOIs of 0.01 and 0.001."
            *   For Sfin-6: "The viable count of S. sonnei 1 cells were also decreased at an MOI of 0.1 and complete lysis occurred within 3.5 h, while MOIs of 0.01 and 0.001 showed complete lysis after 4.5 h (p < 0.005; Figures 2D–F)."

    *   **Parameters Check:**
        *   Primary targeted bacteria species (not Strain): Yes, the text refers to specific strains (2a, 1, 1).
        *   Phage's Plaque characteristics/Shape: Yes, "isometric head (64.90 ± 2.04 nm and 62.42 ± 4.04 nm, respectively) and a non-contractile tail (145 ± 8.5 nm and 148.47 ± 14.5 nm, respectively)". (Sfin-2 and Sfin-6 shared these general characteristics).
        *   Optimal MOI: Specific MOIs (0.1, 0.01, 0.001) are tested against different strains.
        *   Latent period (min): Mentioned as 5–20 min (general). Specific values for Sfin-2/Sfin-6 against individual hosts are given (e.g., Sfin-2 latent period of ~7 min against S. flexneri 2a).
        *   Burst size (phage/infected bacterium): Mentioned as "ranging from ~74 to 265 PFU/cell" (general) and specific values for each host/phage combination (e.g., Sfin-2 against S. sonnei 1: 101 PFU/cell).
        *   Optimal Temperature (°C): Mentions incubation at 37°C (for most tests) and thermal stability tests (37, 50, 60, 70, 80, 90°C).
        *   Optimal pH: Mentions highest activity at pH 7.0 and recovery at pH 4.0 and pH 12.0.

*   Conclusion: The text provides specific, quantitative data (MOI dependence, burst size, latent period, temperature, pH) for the interaction of the phages with these three specific strains, making them suitable for separate rows.V1: Separate
V2: Separate
V3: Separate

In [ ]:
pass3_json_formatting = (
    "You are a strict data formatting assistant\n"
    "DO NOT think. DO NOT write thoughts. DO NOT use the thinking channel or <|channel>thought.\n\n"
    "Convert the following input values into a JSON array.\n\n"

    "Rules:\n"
    "- Output only valid JSON.\n"
    "- Each input line becomes one JSON string.\n"
    "- Never split a line into multiple strings.\n"
    "- Remove the prefix 'Vn:' from each line.\n"
    "- Keep all remaining characters exactly as written, including commas.\n\n"

    "### Example 1\n"

    "Input Values:\n\n"
    "V1: <value>, <value>"

    "\n\nOutput:\n\n"
    "['<value>, <value>']"

    "\n### Example 2\n"

    "Input Values:\n\n"
    "V1: Apple, Banana, Orange\n\n"
    "V2: Cat; Dog; Mouse\n\n"
    "V3: A/B/C"

    "\n\nOutput:\n"
    "['Apple, Banana, Orange', 'Cat; Dog; Mouse', 'A/B/C']\n"

    "### Input Values\n"

    "{pass2_values}"
)

pass2_values = pass1_text
pass2_values = "V1: Siphoviridae, Caudovirales"
pass2_values = pass2_text

pass3_prompt = pass3_json_formatting.format(pass2_values=pass2_values)
chat._messages[:] = chat._messages[:1]
chat.add_user_message(pass3_prompt)

Markdown(pass3_prompt)

In [ ]:
pass3_response = model.respond(pass3_prompt)
pass3_raw = pass3_response.content
pass3_thought, pass3_json_str = split_llm_response(pass3_raw)

Markdown(pass3_thought + pass3_json_str)

In [ ]:
MAX_ITERATIONS = 1
gif_frames = []

for i in range(MAX_ITERATIONS):
    row, column, conditions, example_val = build_conditions_and_example(tbl_extract)
    if row is None or column is None:
        print("Extraction completed!")
        break
        
    chat._messages[:] = chat._messages[:2]
    
    # Pass 1: Fact Extraction
    pass1_prompt = format_extraction_prompt(column, conditions, example_val, pass1_fact_extraction)
    chat.add_user_message(pass1_prompt)
    
    pass1_response = model.respond(chat)
    pass1_raw = pass1_response.content
    pass1_thought, pass1_text = split_llm_response(pass1_raw)
    if not pass1_text:
        pass1_text = pass1_raw
        
    # Check if multiple values were synthesized
    has_multiple_candidates = bool(re.search(r'^V2:\s*', pass1_text, re.MULTILINE | re.IGNORECASE))
    
    if not has_multiple_candidates:
        # Bypassing Pass 2
        pass2_thought = "*Bypassed (Single candidate value detected)*"
        pass2_values = extract_values_from_decision(pass1_text)
        
        # Pass 3: JSON Formatting
        pass3_prompt = pass3_json_formatting.format(pass2_values=pass2_values)
        # chat._messages.append({"role": "assistant", "content": pass1_raw})
        chat.add_assistant_response(pass1_raw)
        chat.add_user_message(pass3_prompt)
        
        pass3_response = model.respond(chat)
        pass3_raw = pass3_response.content
        pass3_thought, pass3_json_str = split_llm_response(pass3_raw)
        if not pass3_json_str:
            pass3_json_str = pass3_raw
    else:
        # Pass 2: Relational Split Decision
        pass2_prompt = pass2_split_decision.format(
            target_column=column,
            pass1_output=pass1_text
        )
        chat._messages.append({"role": "assistant", "content": pass1_raw})
        chat.add_user_message(pass2_prompt)
        
        pass2_response = model.respond(chat)
        pass2_raw = pass2_response.content
        pass2_thought, pass2_text = split_llm_response(pass2_raw)
        if not pass2_text:
            pass2_text = pass2_raw
            
        pass2_values = extract_values_from_decision(pass2_text)
        
        # Pass 3: JSON Formatting
        pass3_prompt = pass3_json_formatting.format(pass2_values=pass2_values)
        # chat._messages.append({"role": "assistant", "content": pass2_raw})
        # chat.add_assistant_response(pass2_raw)
        # chat.add_user_message(pass3_prompt)
        
        # pass3_response = model.respond(chat)
        pass3_response = model.respond(pass3_prompt)
        pass3_raw = pass3_response.content
        pass3_thought, pass3_json_str = split_llm_response(pass3_raw)
        if not pass3_json_str:
            pass3_json_str = pass3_raw
            
    try:
        data = extract_json_array(pass3_json_str)
    except Exception as e:
        print(f"Malformed JSON detected for '{column}'. Falling back.")
        data = [pass3_json_str]
        
    tbl_extract = insert_and_split(tbl_extract, row, column, data)
    
    try:
        frame = df_to_image(tbl_extract)
        gif_frames.append(frame)
    except Exception:
        pass
        
    clear_output(wait=True)
    print(f"Extracted {column}: {data}")
    display(tbl_extract)
    
    combined_thought = (
        f"### FACT EXTRACTION\n{pass1_thought or 'No thoughts recorded.'}\n\n"
        f"### SPLIT DECISION\n{pass2_thought or 'No thoughts recorded.'}\n\n"
    )
    print(combined_thought)

In [ ]:
# Markdown(combined_thought)
print(data)

In [ ]:
from IPython.display import Image as IPImage, display
if gif_frames:
    gif_frames[0].save(
        "table_extraction_progress_multipass.gif",
        save_all=True,
        append_images=gif_frames[1:],
        optimize=False,
        duration=800,
        loop=0
    )

# Display the GIF
display(IPImage(filename="table_extraction_progress_multipass.gif"))

In [ ]:
tbl_extract.to_csv(OUTPUT_CSV_PATH, index=False)